# LAB 2 — Embeddings: BGE-M3 vs Sentence-Transformers, Similaridade Coseno e UMAP

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Duração estimada:** 60 minutos  
**Ambiente:** Jupyter Local no VS Code

---

## Objetivos de Aprendizagem
1. Encodar 40 frases jurídicas com BGE-M3 e sentence-transformers
2. Calcular e comparar matrizes de similaridade coseno
3. Visualizar clusters semânticos em 2D com UMAP
4. Comparar qual modelo é mais adequado para o domínio jurídico-PT
5. Discussão: o que faz um bom embedding para RAG?

---

## Modelos de Embedding no Curso

Este lab demonstra duas abordagens para geração de embeddings:

| Abordagem | Modelo | Onde Roda | Dimensões |
|-----------|--------|-----------|----------|
| Ollama (padrão do curso) | `nomic-embed-text` | Ollama local | 768 |
| Ollama (alternativa) | `mxbai-embed-large` | Ollama local | 1024 |
| sentence-transformers (direto) | `BAAI/bge-m3` | Python direto | 1024 |
| sentence-transformers (leve) | `paraphrase-multilingual-mpnet-base-v2` | Python direto | 768 |

**Quando usar cada abordagem:**
- **Ollama:** produção, integração simples, sem gerenciar modelos Python
- **sentence-transformers direto:** desenvolvimento, fine-tuning, modelos não disponíveis no Ollama

> **Conexão com a Teoria:** Este lab materializa o Tópico 3 da teoria. Você verá visualmente  
> como 'absolvido' e 'inocentado' ficam próximos no espaço vetorial,  
> enquanto 'furto' e 'homicídio' ficam em clusters diferentes.

## 📦 CÉLULA 1 — Instalação e Imports

In [ ]:
# CÉLULA 1: Instalação e Imports
# Usar %pip no Jupyter local garante instalacao no kernel correto
%pip install sentence-transformers FlagEmbedding umap-learn scikit-learn==1.5.2 matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings
warnings.filterwarnings('ignore')

# Configura estilo de plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Detecta dispositivo — GPU se disponivel, CPU caso contrario
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB')
else:
    print('  CPU mode — modelos menores serao mais rapidos')
    print('  Para GPU: instale PyTorch com CUDA (opcional)')

# Configuracao Ollama para embeddings alternativos
OLLAMA_BASE_URL = 'http://localhost:11434'

import requests
try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3)
    modelos_ollama = [m['name'] for m in r.json().get('models', [])]
    print(f'\nOllama disponivel — modelos: {modelos_ollama}')
    OLLAMA_DISPONIVEL = True
except Exception:
    print('\nOllama nao detectado — usando apenas sentence-transformers direto')
    OLLAMA_DISPONIVEL = False


## 📦 CÉLULA 2 — Dataset: 20 Frases Jurídicas Categorizadas

**O que faz:** Define o conjunto de 20 frases jurídicas distribuídas em 4 categorias semânticas.

**Por que:** Para visualizar clusters, precisamos de frases com agrupamentos semânticos conhecidos — assim podemos validar se o embedding capturou corretamente a semântica do domínio.

In [ ]:
# Célula 2: Dataset de frases jurídicas

# 20 frases organizadas em 4 categorias temáticas
# Cada categoria tem 5 frases semanticamente relacionadas
# Célula 2: Dataset de frases jurídicas ampliado
# O que faz: Define 40 frases em 8 categorias temáticas para testar a sensibilidade dos modelos.
# Por que: Mais categorias e nuances testam se o modelo capta termos técnicos ou apenas o "assunto" geral.

dados = [
    # ── CATEGORIA 1: Crimes Patrimoniais (Nuance: Violência x Destreza) ──
    {'frase': 'O acusado subtraiu bens da residência utilizando escalada e destreza.', 'categoria': 'Crimes Patrimoniais', 'cor': '#E74C3C'},
    {'frase': 'Houve furto qualificado mediante o rompimento de obstáculo à subtração da coisa.', 'categoria': 'Crimes Patrimoniais', 'cor': '#E74C3C'},
    {'frase': 'O réu praticou roubo mediante grave ameaça e uso de arma de fogo.', 'categoria': 'Crimes Patrimoniais', 'cor': '#E74C3C'},
    {'frase': 'A vítima foi rendida por dois indivíduos que subtraíram seu veículo em via pública.', 'categoria': 'Crimes Patrimoniais', 'cor': '#E74C3C'},
    {'frase': 'O estelionatário induziu a vítima em erro por meio de artifício ardil e fraude.', 'categoria': 'Crimes Patrimoniais', 'cor': '#E74C3C'},

    # ── CATEGORIA 2: Crimes Contra a Vida (Nuance: Dolo x Culpa) ──
    {'frase': 'O réu agiu com animus necandi ao desferir golpes contra a vítima.', 'categoria': 'Crimes Contra a Vida', 'cor': '#2C3E50'},
    {'frase': 'A denúncia narra homicídio qualificado por motivo fútil e traição.', 'categoria': 'Crimes Contra a Vida', 'cor': '#2C3E50'},
    {'frase': 'O condutor agiu com imprudência, causando homicídio culposo na direção de veículo.', 'categoria': 'Crimes Contra a Vida', 'cor': '#2C3E50'},
    {'frase': 'Houve feminicídio praticado em contexto de violência doméstica e familiar.', 'categoria': 'Crimes Contra a Vida', 'cor': '#2C3E50'},
    {'frase': 'O agente causou lesão corporal de natureza grave resultando em incapacidade.', 'categoria': 'Crimes Contra a Vida', 'cor': '#2C3E50'},

    # ── CATEGORIA 3: Direito Processual Penal (Atos e Medidas) ──
    {'frase': 'O habeas corpus busca cessar o constrangimento ilegal por excesso de prazo.', 'categoria': 'Direito Processual', 'cor': '#27AE60'},
    {'frase': 'O magistrado decretou a custódia preventiva para garantia da ordem pública.', 'categoria': 'Direito Processual', 'cor': '#27AE60'},
    {'frase': 'A audiência de custódia foi realizada dentro do prazo de 24 horas da prisão.', 'categoria': 'Direito Processual', 'cor': '#27AE60'},
    {'frase': 'O Ministério Público ofereceu denúncia com base no inquérito policial.', 'categoria': 'Direito Processual', 'cor': '#27AE60'},
    {'frase': 'O réu foi citado para apresentar resposta à acusação no prazo legal.', 'categoria': 'Direito Processual', 'cor': '#27AE60'},

    # ── CATEGORIA 4: Investigação e Perícia ──
    {'frase': 'A perícia técnica realizou o exame de corpo de delito no local do crime.', 'categoria': 'Investigação e Perícia', 'cor': '#8E44AD'},
    {'frase': 'O laudo papiloscópico confirmou a presença das digitais do suspeito na cena.', 'categoria': 'Investigação e Perícia', 'cor': '#8E44AD'},
    {'frase': 'O delegado de polícia presidiu a diligência de busca e apreensão domiciliar.', 'categoria': 'Investigação e Perícia', 'cor': '#8E44AD'},
    {'frase': 'Cadeia de custódia: a preservação dos vestígios é fundamental para a instrução.', 'categoria': 'Investigação e Perícia', 'cor': '#8E44AD'},
    {'frase': 'A testemunha foi ouvida em sede de inquérito sob o crivo da autoridade policial.', 'categoria': 'Investigação e Perícia', 'cor': '#8E44AD'},

    # ── CATEGORIA 5: Crimes contra a Administração Pública ──
    {'frase': 'O servidor público solicitou vantagem indevida para agilizar o parecer técnico.', 'categoria': 'Crimes Administrativos', 'cor': '#F39C12'},
    {'frase': 'Configura peculato o desvio de bens públicos em proveito do funcionário.', 'categoria': 'Crimes Administrativos', 'cor': '#F39C12'},
    {'frase': 'O gestor municipal prevaricou ao retardar ato de ofício por interesse pessoal.', 'categoria': 'Crimes Administrativos', 'cor': '#F39C12'},
    {'frase': 'O particular ofereceu suborno ao fiscal para evitar a interdição do estabelecimento.', 'categoria': 'Crimes Administrativos', 'cor': '#F39C12'},
    {'frase': 'A concussão ocorre quando o agente público exige, para si, vantagem indevida.', 'categoria': 'Crimes Administrativos', 'cor': '#F39C12'},

    # ── CATEGORIA 6: Crimes Cibernéticos ──
    {'frase': 'O hacker invadiu o dispositivo informático para subtrair dados telemáticos.', 'categoria': 'Crimes Cibernéticos', 'cor': '#1ABC9C'},
    {'frase': 'A vítima sofreu extorsão após ter seus arquivos sequestrados por ransomware.', 'categoria': 'Crimes Cibernéticos', 'cor': '#1ABC9C'},
    {'frase': 'Houve disseminação de código malicioso para captura de senhas bancárias.', 'categoria': 'Crimes Cibernéticos', 'cor': '#1ABC9C'},
    {'frase': 'O ataque de negação de serviço (DDoS) derrubou os servidores do tribunal.', 'categoria': 'Crimes Cibernéticos', 'cor': '#1ABC9C'},
    {'frase': 'Praticou phishing ao criar página falsa para capturar dados de usuários.', 'categoria': 'Crimes Cibernéticos', 'cor': '#1ABC9C'},

    # ── CATEGORIA 7: Crimes Ambientais ──
    {'frase': 'A empresa foi autuada por lançar resíduos tóxicos em corpo d\'água.', 'categoria': 'Crimes Ambientais', 'cor': '#16A085'},
    {'frase': 'Houve desmatamento ilegal em unidade de conservação de proteção integral.', 'categoria': 'Crimes Ambientais', 'cor': '#16A085'},
    {'frase': 'O transporte de madeira sem licença ambiental configura infração penal.', 'categoria': 'Crimes Ambientais', 'cor': '#16A085'},
    {'frase': 'O garimpo ilegal utilizou mercúrio, contaminando o leito do rio amazônico.', 'categoria': 'Crimes Ambientais', 'cor': '#16A085'},
    {'frase': 'Praticar maus-tratos contra animais silvestres é crime previsto em lei específica.', 'categoria': 'Crimes Ambientais', 'cor': '#16A085'},

    # ── CATEGORIA 8: Violência Doméstica / Maria da Penha ──
    {'frase': 'O agressor descumpriu a medida protetiva de urgência fixada pelo juiz.', 'categoria': 'Violência Doméstica', 'cor': '#D35400'},
    {'frase': 'Houve ameaça e injúria contra a ex-companheira no âmbito doméstico.', 'categoria': 'Violência Doméstica', 'cor': '#D35400'},
    {'frase': 'A vítima relatou violência psicológica e cárcere privado pelo agressor.', 'categoria': 'Violência Doméstica', 'cor': '#D35400'},
    {'frase': 'A Lei Maria da Penha visa a prevenção e punição da violência de gênero.', 'categoria': 'Violência Doméstica', 'cor': '#D35400'},
    {'frase': 'O monitoramento eletrônico foi imposto ao agressor por risco à vida da vítima.', 'categoria': 'Violência Doméstica', 'cor': '#D35400'},
]

# Converte os dados para DataFrame e extrai as colunas em listas
import pandas as pd
df = pd.DataFrame(dados)
frases = df['frase'].tolist()
categorias = df['categoria'].tolist()
cores = df['cor'].tolist()

# Exibe o total de frases e a contagem de itens por categoria
print(f'📋 Dataset ampliado: {len(frases)} frases em {df["categoria"].nunique()} categorias')
for cat in df['categoria'].unique():
    n = (df['categoria'] == cat).sum()
    print(f'  📁 {cat}: {n} frases')


# (Abaixo temos apenas a repetição do código acima)
df = pd.DataFrame(dados)
frases = df['frase'].tolist()
categorias = df['categoria'].tolist()
cores = df['cor'].tolist()

print(f'📋 Dataset criado: {len(frases)} frases em {df["categoria"].nunique()} categorias')
print()
for cat in df['categoria'].unique():
    n = (df['categoria'] == cat).sum()
    print(f'  📁 {cat}: {n} frases')


## 📦 CÉLULA 3 — Carregar Modelos de Embedding

**O que faz:** Carrega os dois modelos de embedding que vamos comparar.

**Por que:** Compararemos BGE-M3 (multilíngue, estado da arte 2024) vs `paraphrase-multilingual-mpnet-base-v2` (sentence-transformers, mais leve) para o domínio jurídico em português.

In [ ]:
# CÉLULA 3: Carregar Modelos de Embedding
# Estrategia: BGE-M3 (estado da arte, ~570MB) vs modelo leve multilingual
# Na primeira execucao, os modelos sao baixados do HuggingFace e cacheados localmente
%pip install hf_xet os
import os 

    # os.environ["HF_TOKEN"] = "SEU_TOKEN_AQUI"  # Opcional: BGE-M3 não exige token

from sentence_transformers import SentenceTransformer

modelos_config = [
    {
        'nome': 'BGE-M3',
        'id_huggingface': 'BAAI/bge-m3',
        'descricao': 'Estado da arte multilíngue (2024), 1024 dims, 8192 tokens max',
        'tamanho': '~570MB',
        'quando_usar': 'Producao: melhor qualidade para PT-BR juridico'
    },
    {
        'nome': 'MiniLM-Multilingual',
        'id_huggingface': 'paraphrase-multilingual-mpnet-base-v2',
        'descricao': 'Leve e eficiente, 768 dims, ideal para prototipagem',
        'tamanho': '~420MB',
        'quando_usar': 'Prototipagem: mais rapido, menor qualidade'
    },
]

modelos_carregados = {}

for cfg in modelos_config:
    print(f'\nCarregando {cfg["nome"]}...')
    print(f'  Modelo: {cfg["id_huggingface"]}')
    print(f'  Descricao: {cfg["descricao"]}')
    print(f'  Tamanho: {cfg["tamanho"]}')

    inicio = time.time()
    try:
        modelo = SentenceTransformer(cfg['id_huggingface'], device=DEVICE)
        modelos_carregados[cfg['nome']] = modelo
        print(f'  OK — Carregado em {time.time() - inicio:.1f}s')
    except Exception as e:
        print(f'  ERRO: {e}')
        print(f'  Verifique conexao com internet para baixar o modelo')

print(f'\nModelos prontos: {list(modelos_carregados.keys())}')

# Alternativa via Ollama (se disponivel)
if OLLAMA_DISPONIVEL:
    print()
    print('Ollama tambem disponivel para embeddings:')
    print('  nomic-embed-text  — pull: ollama pull nomic-embed-text')
    print('  mxbai-embed-large — pull: ollama pull mxbai-embed-large')
    print('  Vantagem: sem gerenciar modelo Python, API REST simples')


## 📦 CÉLULA 4 — Gerar Embeddings e Calcular Similaridade

**O que faz:** Gera embeddings para as 80 frases com cada modelo e calcula a matriz de similaridade coseno.

**Por que:** A comparação das matrizes de similaridade revela qual modelo captura melhor a semântica jurídica.

In [ ]:
import time
import numpy as np
import requests
from sklearn.metrics.pairwise import cosine_similarity
def gerar_embeddings(modelos, frases, usar_ollama=False, ollama_url='http://localhost:11434'):
    """
    Gera embeddings e calcula similaridade coseno para modelos locais (Hugging Face)
    ou modelos rodando no Ollama Server.
    
    Parâmetros:
    - modelos: Dicionário 'modelos_carregados' (para usar_ollama=False)
               OU Lista de nomes de modelos ['nomic-embed-text'] (para usar_ollama=True).
    - frases: Lista contendo as frases a serem processadas.
    - usar_ollama: Define se vai usar a API do Ollama (True) ou rodar localmente no Python (False).
    - ollama_url: Endereço do seu servidor local do Ollama.
    """
    resultados = {}
    
    # ── MODO 1: GERAR COM OLLAMA ──
    if usar_ollama:
        for nome_modelo in modelos:
            print(f'\n🔢 Gerando embeddings com Ollama ({nome_modelo})...')
            inicio = time.time()
            try:
                response = requests.post(
                    f'{ollama_url}/api/embed',
                    json={'model': nome_modelo, 'input': frases},
                    timeout=30
                )
                if response.status_code != 200:
                    raise Exception(f"Erro na API do Ollama: {response.text}")
                    
                embeddings = np.array(response.json().get('embeddings', []))
                
                # Normalização L2 para correta similaridade coseno via produto interno
                norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
                norms = np.where(norms == 0, 1e-9, norms)
                embeddings = embeddings / norms
                
                tempo = time.time() - inicio
                sim_matrix = cosine_similarity(embeddings)
                
                resultados[nome_modelo] = {
                    'embeddings': embeddings,
                    'sim_matrix': sim_matrix,
                    'dimensoes': embeddings.shape[1],
                    'tempo_s': tempo
                }
                print(f'   ✅ Dimensões: {embeddings.shape} | ⏱️ Tempo: {tempo:.2f}s')
            except Exception as e:
                print(f'   ❌ Erro no modelo Ollama "{nome_modelo}": {e}')
                
    # ── MODO 2: GERAR SEM OLLAMA (Modelos locais direct-Python) ──
    else:
        for nome, modelo in modelos.items():
            print(f'\n🔢 Gerando embeddings locais com {nome}...')
            inicio = time.time()
            try:
                # encode() da biblioteca sentence-transformers
                embeddings = modelo.encode(
                    frases,
                    batch_size=8,
                    show_progress_bar=True,
                    normalize_embeddings=True,
                    convert_to_numpy=True
                )
                tempo = time.time() - inicio
                sim_matrix = cosine_similarity(embeddings)
                
                resultados[nome] = {
                    'embeddings': embeddings,
                    'sim_matrix': sim_matrix,
                    'dimensoes': embeddings.shape[1],
                    'tempo_s': tempo
                }
                print(f'   ✅ Dimensões: {embeddings.shape} | ⏱️ Tempo: {tempo:.2f}s')
            except Exception as e:
                print(f'   ❌ Erro no modelo local "{nome}": {e}')
                
    return resultados

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

def plotar_heatmaps_similaridade(resultados, dataframe):
    """
    Gera e plota mapas de calor (heatmaps) de similaridade para QUALQUER modelo
    (seja do Ollama, SentenceTransformers, BGE-M3, etc).
    
    Parâmetros:
    - resultados: Dicionário contendo os resultados dos embeddings.
    - dataframe: O DataFrame Pandas contendo frases, categorias e cores.
    """
    if not resultados:
        print("❌ Nenhum resultado de embedding disponível para plotar.")
        return
        
    fig, axes = plt.subplots(1, len(resultados), figsize=(24, 12))
    if len(resultados) == 1:
        axes = [axes]
        
    labels_curtos = [
        f"{cat} ({frase[:15]}...)"
        for cat, frase in zip(dataframe['categoria'], dataframe['frase'])
    ]
    
    categorias_unicas = dataframe['categoria'].unique()
    total_frases = len(dataframe)
    
    for idx, (nome, res) in enumerate(resultados.items()):
        ax = axes[idx]
        sim = res['sim_matrix']
        
        sns.heatmap(
            sim,
            ax=ax,
            xticklabels=labels_curtos,
            yticklabels=labels_curtos,
            cmap='RdYlGn',
            vmin=0, vmax=1,
            annot=False,
            fmt='.2f',
            linewidths=0.05,
            linecolor='lightgray'
        )
        
        # Título dinâmico que funciona para qualquer tipo de modelo
        ax.set_title(
            f'{nome}\n(dim={res["dimensoes"]}, tempo={res["tempo_s"]:.1f}s)',
            fontsize=14, fontweight='bold', pad=12
        )
        ax.tick_params(axis='x', rotation=90, labelsize=6)
        ax.tick_params(axis='y', rotation=0, labelsize=6)
        
        for idx_cat, i in enumerate(range(0, total_frases, 5)):
            cor_categoria = dataframe[dataframe['categoria'] == categorias_unicas[idx_cat]]['cor'].iloc[0]
            ax.add_patch(plt.Rectangle((i, i), 5, 5,
                                       fill=False, edgecolor=cor_categoria,
                                       lw=2.5, linestyle='--'))
            
    patches = [mpatches.Patch(color=dataframe[dataframe['categoria']==cat]['cor'].iloc[0],
                              label=cat)
               for cat in categorias_unicas]
               
    fig.legend(handles=patches, loc='lower center', ncol=4,
               fontsize=10, title='Categorias', bbox_to_anchor=(0.5, -0.06))
               
    plt.suptitle(
        'Matrizes de Similaridade Coseno — Comparativo de Modelos (Dataset Ampliado)\n'
        '(Quadrados tracejados coloridos = Fronteiras semânticas de cada categoria)',
        fontsize=16, fontweight='bold', y=1.04
    )
    
    plt.tight_layout()
    plt.savefig('similaridade_modelos_comparativo.png', dpi=150, bbox_inches='tight')
    plt.show()


## 📦 CÉLULA 5 — Visualização das Matrizes de Similaridade

**O que faz:** Plota heatmaps das matrizes de similaridade coseno para cada modelo.

**Por que:** Visualizações revelam padrões que números isolados não mostram — clusters de alta similaridade devem aparecer em blocos na diagonal.

In [ ]:
# Roda o BGE-M3 e MiniLM direto no Python
resultados_locais = gerar_embeddings(modelos_carregados, frases, usar_ollama=False)
# Plota o gráfico deles
plotar_heatmaps_similaridade(resultados_locais, df)

In [ ]:
# Roda os modelos do seu Ollama local
modelos_para_testar = ['nomic-embed-text','bge-m3','paraphrase-multilingual']
resultados_ollama = gerar_embeddings(modelos_para_testar, frases, usar_ollama=True)
# Plota o gráfico deles
plotar_heatmaps_similaridade(resultados_ollama, df)

## 📦 CÉLULA 6 — Redução Dimensional com UMAP

**O que faz:** Reduz os embeddings de 768/1024 dimensões para 2D usando UMAP, permitindo visualização dos clusters semânticos.

**Por que:** UMAP preserva melhor a estrutura global dos dados que t-SNE, sendo mais adequado para visualizar relações semânticas em corpora jurídicos.

In [ ]:
# Célula 6: Função de Visualização UMAP Genérica
import umap
import matplotlib.pyplot as plt
import numpy as np

def plotar_umap_similaridade(resultados, dataframe):
    """
    Gera e plota projeções UMAP em 2D para visualização dos clusters semânticos
    dos embeddings. Funciona para modelos Ollama e locais (SentenceTransformers).
    
    Parâmetros:
    - resultados: Dicionário contendo os embeddings gerados.
    - dataframe: O DataFrame Pandas contendo frases, categorias e cores.
    """
    if not resultados:
        print("❌ Nenhum resultado de embedding disponível para projetar com UMAP.")
        return
        
    # Gera rótulos para as frases (primeiros 30 caracteres de cada frase)
    labels_curtos = [f[:30] + "..." for f in dataframe['frase']]
    
    # Configura a área de desenho com base na quantidade de modelos
    fig, axes = plt.subplots(1, len(resultados), figsize=(22, 10))
    if len(resultados) == 1:
        axes = [axes]
        
    # Itera sobre cada modelo contido no dicionário de resultados
    for idx, (nome, res) in enumerate(resultados.items()):
        ax = axes[idx]
        embeddings = res['embeddings']
        
        print(f'\n🗺️  Calculando UMAP para o modelo: {nome}...')
        
        # Instancia e configura o redutor UMAP (redução para 2 dimensões)
        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=5,      
            min_dist=0.1,
            metric='cosine',    
            random_state=42     
        )
        
        # Reduz os embeddings (ex: 768 ou 1024 dimensões) para coordenadas X, Y 2D
        embeddings_2d = reducer.fit_transform(embeddings)
        
        # Desenha os pontos coloridos por categoria
        for cat in dataframe['categoria'].unique():
            mask = dataframe['categoria'] == cat
            pontos = embeddings_2d[mask.values]
            cor = dataframe[mask]['cor'].iloc[0]
            
            # Desenha as bolhas dispersas no gráfico 2D
            ax.scatter(
                pontos[:, 0], pontos[:, 1],
                c=cor, s=200, alpha=0.7,
                edgecolors='white', linewidth=1,
                label=cat, zorder=3
            )
            
            # Adiciona anotação textual do início da frase sobre cada ponto
            indices_cat = dataframe[mask].index.tolist()
            for i_local, i_global in enumerate(indices_cat):
                label_texto = labels_curtos[i_global]
                ax.annotate(
                    label_texto,
                    (pontos[i_local, 0], pontos[i_local, 1]),
                    textcoords='offset points',
                    xytext=(5, 5),
                    fontsize=8,
                    alpha=0.8
                )
        
        # Configurações estéticas de cada gráfico individual
        ax.set_title(f'Clusterização Semântica (UMAP) - {nome}', fontsize=15, fontweight='bold')
        ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
        ax.set_xlabel('Dimensão UMAP 1')
        ax.set_ylabel('Dimensão UMAP 2')
        
    plt.tight_layout()
    # Salva a imagem resultante localmente
    plt.savefig('clusterizacao_umap_modelos.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Exibe diretrizes de interpretação
    print('\n💡 INTERPRETAÇÃO DO GRÁFICO UMAP:')
    print('   → Clusters bem separados = bom embedding para RAG')
    print('   → Pontos da mesma categoria agrupados = excelente captura semântica')
    print('   → Frases de categorias diferentes distantes = alta discriminação semântica')


In [ ]:
plotar_umap_similaridade(resultados_locais, df)

In [ ]:
plotar_umap_similaridade(resultados_ollama, df)


## 📦 CÉLULA 7 — Análise Quantitativa: Top-5 Vizinhos Mais Próximos

**O que faz:** Para uma frase de query, encontra as 5 frases mais semanticamente similares em cada modelo.

**Por que:** Esta é exatamente a operação que um sistema RAG executa na fase de retrieval.

In [ ]:
# Célula 7: Função Reutilizável de Simulação de Retrieval
import requests
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def simular_retrieval_embeddings(resultados, dataframe, queries):
    """
    Simula o processo de busca (retrieval) exibindo os Top-5 vizinhos mais próximos.
    Funciona tanto para modelos locais (SentenceTransformers) quanto para os do Ollama.
    
    Parâmetros:
    - resultados: O dicionário de resultados que você quer testar (resultados_locais ou resultados_ollama).
    - dataframe: O DataFrame Pandas 'df' contendo frases, categorias e cores.
    - queries: Lista de strings contendo os termos de busca para o teste.
    """
    if not resultados:
        print("❌ Nenhum resultado de embedding disponível para simular o retrieval.")
        return
        
    print('🔍 SIMULAÇÃO DE RETRIEVAL — Top-5 Vizinhos')
    print('=' * 70)
    
    # Busca dinamicamente os modelos locais carregados no ambiente do Jupyter
    modelos_locais = globals().get('modelos_carregados', {})
    
    for query in queries:
        print(f'\n📥 QUERY: "{query}"')
        print('-' * 70)
        
        # Avalia apenas os modelos contidos no dicionário enviado para a função
        for nome, res in resultados.items():
            embeddings = res['embeddings']
            
            # DETECÇÃO AUTOMÁTICA (Local ou Ollama)
            if nome in modelos_locais:
                # Caso 1: Encodagem Local (SentenceTransformers)
                modelo = modelos_locais[nome]
                query_emb = modelo.encode(
                    [query],
                    normalize_embeddings=True,
                    convert_to_numpy=True
                )
            else:
                # Caso 2: Encodagem via API do Ollama Server
                try:
                    response = requests.post(
                        'http://localhost:11434/api/embed',
                        json={'model': nome, 'input': [query]},
                        timeout=10
                    )
                    if response.status_code != 200:
                        raise Exception(f"Erro na API do Ollama: {response.text}")
                        
                    query_emb = np.array(response.json().get('embeddings', []))
                    
                    # Normalização L2 para correta similaridade cosseno
                    norms = np.linalg.norm(query_emb, axis=1, keepdims=True)
                    norms = np.where(norms == 0, 1e-9, norms)
                    query_emb = query_emb / norms
                    
                except Exception as e:
                    print(f'  ❌ Erro ao gerar embedding da query com o modelo Ollama "{nome}": {e}')
                    continue
            
            # Calcula similaridade coseno da query com os documentos
            sims = cosine_similarity(query_emb, embeddings)[0]
            
            # Top-5 vizinhos mais similares
            top5_indices = np.argsort(sims)[::-1][:5]
            
            print(f'\n  🤖 {nome}:')
            for rank, idx in enumerate(top5_indices, 1):
                categoria = dataframe.iloc[idx]['categoria']
                sim = sims[idx]
                frase_curta = dataframe.iloc[idx]['frase'][:65]
                print(f'    {rank}. [{sim:.3f}] [{categoria[:20]}] {frase_curta}...')
                
    print('\n' + '=' * 70)


In [ ]:
queries_teste = [
    'busca por documentos sobre subtração de patrimônio',
    'pesquisa sobre prisão cautelar e garantias processuais',
    'como foi investigado o crime'
]

In [ ]:
simular_retrieval_embeddings(resultados_locais, df, queries_teste)


In [ ]:
simular_retrieval_embeddings(resultados_ollama, df, queries_teste)


In [ ]:
# Nova Célula: Função de Gráfico Comparativo de Desempenho
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plotar_comparativo_scores(resultados, dataframe):
    """
    Gera um gráfico de barras agrupadas comparando a similaridade média
    intra-categoria de cada modelo. Adiciona rótulos de dados nas barras.
    
    Parâmetros:
    - resultados: Dicionário contendo os embeddings e similaridades.
    - dataframe: O DataFrame Pandas 'df'.
    """
    if not resultados:
        print("❌ Nenhum resultado disponível para gerar a comparação.")
        return
        
    dados_plot = []
    categorias = dataframe['categoria'].unique()
    
    # Extrai e calcula os scores de cada modelo dinamicamente
    for nome_modelo, res in resultados.items():
        sim_matrix = res['sim_matrix']
        for cat in categorias:
            # Filtra os índices daquela categoria
            indices = dataframe[dataframe['categoria'] == cat].index.tolist()
            # Calcula a similaridade média interna (excluindo si mesmo)
            sims = [sim_matrix[i, j] 
                    for i in indices for j in indices if i != j]
            sim_media = np.mean(sims)
            
            # Guarda no formato ideal para o Seaborn plotar
            dados_plot.append({
                'Modelo': nome_modelo,
                'Categoria': cat,
                'Similaridade Média': sim_media
            })
            
    # Converte a lista em um DataFrame temporário
    df_scores = pd.DataFrame(dados_plot)
    
    # Configura o tamanho e estilo do gráfico
    plt.figure(figsize=(18, 9))
    sns.set_style('whitegrid')
    
    # Paleta de cores moderna e profissional
    paleta = sns.color_palette("muted", len(resultados))
    
    # Cria o gráfico de barras agrupadas
    ax = sns.barplot(
        data=df_scores,
        x='Categoria',
        y='Similaridade Média',
        hue='Modelo',
        palette=paleta,
        edgecolor='black',
        linewidth=0.7
    )
    
    # ADICIONA OS SCORES EXATOS NO TOPO DAS BARRAS
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=4, fontsize=9, fontweight='bold', color='black')
        
    # Ajustes estéticos e títulos
    plt.title('Comparativo de Similaridade Semântica Média por Categoria\n'
              '(Barras mais altas = melhor capacidade de agrupamento no RAG)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Categorias do Corpus Jurídico', fontsize=12, fontweight='bold', labelpad=15)
    plt.ylabel('Similaridade Coseno Média', fontsize=12, fontweight='bold', labelpad=15)
    plt.xticks(rotation=25, ha='right', fontsize=10)
    plt.ylim(0, 1.0) # Escala de 0.0 (sem similaridade) a 1.0 (similaridade perfeita)
    
    # Legenda limpa
    plt.legend(title='Modelos Analisados', title_fontsize=11, loc='upper right', frameon=True)
    
    plt.tight_layout()
    # Salva o comparativo visual no seu computador
    plt.savefig('comparativo_scores_modelos.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Cria um dicionário único contendo TODOS os modelos testados (Locais e Ollama)
resultados_totais = {}
if 'resultados_locais' in globals() and resultados_locais:
    resultados_totais.update(resultados_locais)
if 'resultados_ollama' in globals() and resultados_ollama:
    resultados_totais.update(resultados_ollama)

# Chama a função para desenhar o gráfico comparativo
plotar_comparativo_scores(resultados_totais, df)


## 📦 CÉLULA 8 — Comparação Final e Decisão: Qual Modelo Usar?

**O que faz:** Compara os dois modelos em métricas relevantes para sistemas RAG jurídicos.

**Por que:** A decisão do modelo de embedding é uma das mais impactantes no pipeline RAG.

In [ ]:
# Célula 8: Comparação final dos modelos

print('📊 COMPARAÇÃO FINAL DOS MODELOS DE EMBEDDING')
print('=' * 70)

# Métricas calculadas
metricas = {}

for nome, res in resultados_embedding.items():
    sim_matrix = res['sim_matrix']
    
    # Métrica 1: Separação intra vs inter categoria
    intra_sims = []  # Frases da mesma categoria
    inter_sims = []  # Frases de categorias diferentes
    
    for i in range(len(frases)):
        for j in range(i + 1, len(frases)):
            sim = sim_matrix[i, j]
            if df.iloc[i]['categoria'] == df.iloc[j]['categoria']:
                intra_sims.append(sim)
            else:
                inter_sims.append(sim)
    
    separacao = np.mean(intra_sims) - np.mean(inter_sims)
    
    metricas[nome] = {
        'Dimensões': res['dimensoes'],
        'Tempo encode (s)': f"{res['tempo_s']:.2f}",
        'Sim. Intra-categoria': f"{np.mean(intra_sims):.3f}",
        'Sim. Inter-categoria': f"{np.mean(inter_sims):.3f}",
        'Separação semântica': f"{separacao:.3f}",
        'Velocidade (fr/s)': f"{len(frases)/res['tempo_s']:.1f}"
    }

# Tabela comparativa
df_metricas = pd.DataFrame(metricas).T
print(df_metricas.to_string())

print('\n' + '=' * 70)
print('🏆 RECOMENDAÇÃO PARA SISTEMAS RAG JURÍDICOS:')
print()
print('  📌 BGE-M3 (BAAI/bge-m3):')
print('     ✅ Melhor qualidade semântica em português')
print('     ✅ Suporte nativo a 100+ idiomas')
print('     ✅ 8192 tokens (petições longas sem truncamento)')
print('     ✅ Busca densa + esparsa + multi-vetor no mesmo modelo')
print('     ⚠️  Mais lento (~570MB, 1024 dims)')
print('     → USE PARA: produção, corpora > 10k documentos')
print()
print('  📌 MiniLM-Multilingual:')
print('     ✅ Rápido e leve (~420MB, 768 dims)')
print('     ✅ Boa qualidade para prototipação')
print('     ⚠️  512 tokens máximos (trunca documentos longos)')
print('     ⚠️  Qualidade inferior em textos jurídicos técnicos')
print('     → USE PARA: protótipos, testes, datasets pequenos')

print('\n✅ Lab 5 concluído! Objetivo cumprido: visualizou clusters semânticos')
print('   com embeddings em textos jurídicos brasileiros.')

In [ ]:
# Célula 8: Comparação final com Análise Detalhada de Prós & Contras para os alunos
import pandas as pd
import numpy as np

print('📊 COMPARAÇÃO FINAL DOS MODELOS DE EMBEDDING')
print('=' * 95)

# 1. UNIFICA OS DICIONÁRIOS DE FORMA SEGURA
resultados_totais = {}
if 'resultados_locais' in globals() and resultados_locais:
    resultados_totais.update(resultados_locais)
if 'resultados_ollama' in globals() and resultados_ollama:
    resultados_totais.update(resultados_ollama)

if not resultados_totais:
    print("❌ Nenhum resultado de embedding disponível para comparar na Célula 8.")
else:
    metricas = {}
    separacao_bruta = {}

    # 2. CALCULA AS MÉTRICAS DE CADA MODELO
    for nome, res in resultados_totais.items():
        sim_matrix = res['sim_matrix']
        
        intra_sims = []  # Similaridade entre frases da mesma categoria (Coesão)
        inter_sims = []  # Similaridade entre frases de categorias diferentes (Ruído)
        
        for i in range(len(frases)):
            for j in range(i + 1, len(frases)):
                sim = sim_matrix[i, j]
                if df.iloc[i]['categoria'] == df.iloc[j]['categoria']:
                    intra_sims.append(sim)
                else:
                    inter_sims.append(sim)
        
        # Separação Semântica = Coesão - Ruído
        separacao = np.mean(intra_sims) - np.mean(inter_sims)
        separacao_bruta[nome] = separacao
        
        metricas[nome] = {
            'Dimensões': res['dimensoes'],
            'Tempo encode (s)': f"{res['tempo_s']:.2f}",
            'Sim. Intra-categoria': f"{np.mean(intra_sims):.3f}",
            'Sim. Inter-categoria': f"{np.mean(inter_sims):.3f}",
            'Separação semântica': f"{separacao:.3f}",
            'Velocidade (fr/s)': f"{len(frases)/res['tempo_s']:.1f}"
        }

    # 3. EXIBE A TABELA COMPARATIVA
    df_metricas = pd.DataFrame(metricas).T
    print(df_metricas.to_string())
    print('=' * 95)

    # 4. DETERMINA E EXPLICA O MODELO VENCEDOR PEDAGOGICAMENTE
    melhor_modelo = max(separacao_bruta, key=separacao_bruta.get)
    melhor_score = separacao_bruta[melhor_modelo]

    print(f'🏆 VENCEDOR DA AVALIAÇÃO: **{melhor_modelo}** (Score de Separação: {melhor_score:.3f})')
    print('-' * 95)
    print('🧠 AULA PRÁTICA: COMO INTERPRETAR ESSES NÚMEROS NO DOMÍNIO JURÍDICO?')
    print()
    print('1️⃣ O que significa "Similaridade Intra-categoria"?')
    print('   - É a força de ATRAÇÃO. Mede o quão bem o modelo agrupa conceitos do mesmo tema.')
    print('   - Exemplo: O quão próximo o modelo coloca "furto mediante escalada" e "fraude do estelionato".')
    print('   - Atenção! Esse valor NUNCA deve ser 1.000. Por quê? Porque furto e estelionato são crimes')
    print('     diferentes! O modelo ideal deve saber que eles são do mesmo tema (Crimes Patrimoniais),')
    print('     mas mantendo uma distância saudável entre suas nuances específicas.')
    print()
    print('2️⃣ O que significa "Similaridade Inter-categoria"?')
    print('   - É a força de CONFUSÃO (Ruído). Mede o quão parecidos o modelo acha que assuntos totalmente')
    print('     diferentes são.')
    print('   - Exemplo: O quão próximo ele colocaria "habeas corpus" (Processual) de "desmatamento ilegal" (Ambiental).')
    print('   - Queremos que esse valor seja o MENOR possível (próximo de 0.000). Se for alto, o modelo é confuso.')
    print()
    print('3️⃣ O critério de desempate de ouro: A "Separação Semântica"')
    print('   - É a margem de segurança matemática: Similaridade Intra-categoria MENOS Similaridade Inter-categoria.')
    print(f'   - O modelo **{melhor_modelo}** venceu porque maximizou essa distância semântica!')
    print('   - Na prática de um sistema RAG, um modelo com alta separação garante que, quando o usuário buscar')
    print('     sobre "prisão preventiva", o sistema traga peças de Processo Penal, e nunca de Crimes Ambientais.')
    print('     Isso mitiga drasticamente a alucinação da LLM por falta de contexto correto.')
    print('=' * 95)

    # 5. ANÁLISE DETALHADA DE PRÓS E CONTRAS DE CADA MODELO (SWOT DOS RESULTADOS)
    print('🔍 ANÁLISE INDIVIDUAL DE DESEMPENHO (PRÓS & CONTRAS):')
    print()
    
    # --- MODELO 1: BGE-M3 ---
    if 'BGE-M3' in metricas:
        m = metricas['BGE-M3']
        print(f'🤖 MODELO: BGE-M3 (BAAI/bge-m3)')
        print(f'   📊 Resultados do Teste: Sim. Intra-cat: {m["Sim. Intra-categoria"]} | Separação: {m["Separação semântica"]} | Vel: {m["Velocidade (fr/s)"]} fr/s')
        print(f'   👍 PRÓS:')
        print(f'      - Alta Coesão: Apresentou um dos maiores índices de similaridade intra-categoria do teste.')
        print(f'      - Separação Semântica Excepcional: Mapeia conceitos jurídicos em clusters geometricamente isolados,')
        print(f'        evitando ruídos e garantindo que buscas por temas diferentes não se misturem no RAG.')
        print(f'      - Arquitetura Robusta: Suporta 8.192 tokens (lê petições e acórdãos inteiros sem fazer cortes ruins).')
        print(f'   👎 CONTRAS:')
        print(f'      - Lentidão na Inferência: Devido aos seus 1024 dimensões e tamanho (~570MB), sua velocidade de')
        print(f'        codificação ({m["Velocidade (fr/s)"]} frases/s) é a menor do teste. Exige GPUs ou CPUs mais robustas.')
        print()

    # --- MODELO 2: MiniLM-Multilingual ---
    if 'MiniLM-Multilingual' in metricas:
        m = metricas['MiniLM-Multilingual']
        print(f'🤖 MODELO: MiniLM-Multilingual (paraphrase-multilingual-mpnet-base-v2)')
        print(f'   📊 Resultados do Teste: Sim. Intra-cat: {m["Sim. Intra-categoria"]} | Separação: {m["Separação semântica"]} | Vel: {m["Velocidade (fr/s)"]} fr/s')
        print(f'   👍 PRÓS:')
        print(f'      - Velocidade Incrível: Processamento extremamente veloz ({m["Velocidade (fr/s)"]} frases/s).')
        print(f'      - Ultra Leve: Ocupa pouquíssima memória (~420MB), ideal para rodar local em qualquer CPU de entrada.')
        print(f'   👎 CONTRAS:')
        print(f'      - Pior Separação Semântica: Com apenas {m["Separação semântica"]} de margem, ele mistura facilmente')
        print(f'        assuntos diferentes. Ele tem muita dificuldade em diferenciar jargões jurídicos técnicos.')
        print(f'      - Janela Limitada (512 tokens): Trunca acórdãos ou petições longas, ignorando o final dos documentos.')
        print()

    # --- MODELO 3: Modelos via Ollama (nomic-embed-text / mxbai-embed-large) ---
    # Captura dinamicamente qualquer outro modelo que tenha sido testado via Ollama
    modelos_ollama_detectados = [nome for nome in metricas if nome not in ['BGE-M3', 'MiniLM-Multilingual']]
    for nome_ollama in modelos_ollama_detectados:
        m = metricas[nome_ollama]
        print(f'🤖 MODELO OLLAMA: {nome_ollama}')
        print(f'   📊 Resultados do Teste: Sim. Intra-cat: {m["Sim. Intra-categoria"]} | Separação: {m["Separação semântica"]} | Vel: {m["Velocidade (fr/s)"]} fr/s')
        print(f'   👍 PRÓS:')
        print(f'      - Arquitetura Moderna: Oferece contexto ampliado (ex: nomic suporta 8.192 tokens) de forma leve.')
        print(f'      - Desacoplamento de Servidor: Roda via API local do Ollama, economizando recursos de RAM do script Python.')
        print(f'      - Desempenho Equilibrado: Apresenta velocidade muito satisfatória com qualidade intermediária.')
        print(f'   👎 CONTRAS:')
        print(f'      - Falta de Calibração Jurídica: Por ser um modelo mais generalista, o score de separação ({m["Separação semântica"]})')
        print(f'        costuma ficar abaixo do BGE-M3. Pode confundir nuances de termos processuais muito técnicos em português.')
        print()

    print('=' * 95)

print('\n📋 DIRETRIZES FINAIS PARA A TOMADA DE DECISÃO NA ARQUITETURA:')
print('  - USE BGE-M3 em produção de RAG Jurídico se exatidão técnica e leitura de arquivos longos forem inegociáveis.')
print('  - USE NOMIC/OLLAMA se precisar de um excelente equilíbrio entre desempenho, facilidade de microsserviços e velocidade.')
print('  - USE MINILM apenas para protótipos rápidos, testes locais ou depuração sem internet.')

print('\n✅ Laboratório concluído com sucesso!')


---

# 🧪 PARTE 2 — Avaliação Quantitativa Avançada de Embeddings

Até aqui comparamos os modelos qualitativamente (heatmaps, UMAP, top-5).
Mas para **decisões de produção em sistemas RAG jurídicos**, precisamos de métricas numéricas reprodutíveis.

Nesta Parte 2 vamos:

| Etapa | Técnica | O que mede |
|-------|---------|-----------|
| 1 | **Ground Truth** | Construção de 20 queries jurídicas com categoria-alvo conhecida |
| 2 | **Hit Rate@K / Recall@K** | Proporção de queries cujo top-K contém ≥ 1 doc relevante |
| 3 | **MRR** (Mean Reciprocal Rank) | Quão alto no ranking aparece o primeiro doc relevante |
| 4 | **NDCG@K** | Qualidade do ranking com relevância graduada (0-2) |
| 5 | **UMAP com queries sobrepostas** | Visualizar onde queries caem em relação aos docs |
| 6 | **Histograma Pos vs Neg + AUC** | Capacidade do modelo de separar relevantes/irrelevantes |
| 7 | **Dashboard consolidado** | Tabela mestre + radar chart |

> **Conexão com a teoria:** estas métricas são padrão na literatura de Information Retrieval (TREC, BEIR, MTEB)
> e serão usadas em todas as aulas subsequentes do MBA (RAGAS — Aula 5).


## 📦 CÉLULA 10 — Ground Truth: Queries Jurídicas com Categoria-Alvo

**O que faz:** Define 20 queries jurídicas (2-3 por categoria) com a **categoria-alvo conhecida** e **categorias tematicamente próximas** (para escala NDCG 0-2).

**Por que:** Para calcular Hit Rate, MRR e NDCG precisamos saber, *a priori*, quais documentos são relevantes para cada query. Como o dataset tem 8 categorias bem-definidas, usamos **relevância por categoria**:

| Grau | Significado |
|------|-------------|
| 2 | Doc da MESMA categoria da query (altamente relevante) |
| 1 | Doc de categoria tematicamente próxima (parcialmente relevante) |
| 0 | Doc de categoria não relacionada (irrelevante) |

**Afinidades temáticas definidas:**
- Crimes Patrimoniais ↔ Crimes Cibernéticos *(ambos envolvem subtração/fraude)*
- Crimes Contra a Vida ↔ Violência Doméstica *(violência física)*
- Direito Processual ↔ Investigação e Perícia *(fases processuais)*
- Crimes Administrativos ↔ Crimes Ambientais *(administração pública)*

> 💡 As queries usam vocabulário **diferente** das frases originais para evitar overfitting lexical e simular busca realista.

### 📝 Gabarito de Avaliação: Entendendo a Saída da Célula
---

#### 1. Resumo das Queries de Avaliação
* **`Ground truth construído: 20 queries`**: Criamos 20 perguntas simuladas de usuários para testar o sistema. Elas utilizam sinônimos e palavras diferentes do banco de dados principal para evitar o *overfitting lexical* (decoreba de palavras-chave) e forçar os modelos a usarem pura semântica.
* **`Categorias cobertas: 8`**: Nossas perguntas cobrem todas as 8 categorias do nosso corpus jurídico ampliado.

---

#### 2. A Matriz de Relevâncias: `shape=(20, 40)`
A matriz de relevâncias é uma tabela de cruzamento (pergunta vs. documento). Ela possui **20 linhas** (uma para cada query de busca de teste) e **40 colunas** (uma para cada frase do nosso banco de dados).

* **`Docs com grau 2 (média): 5.0 por query`** 🌟
  * *O que significa:* Cada pergunta tem, em média, exatamente 5 documentos perfeitamente relevantes (nota 2) no banco.
  * *Explicação matemática:* Como cada uma das 8 categorias possui exatamente **5 frases** no nosso dataset ampliado, qualquer pergunta de teste terá como resposta ideal exatamente essas 5 frases da mesma categoria alvo.
  
* **`Docs com grau 1 (média): 5.0 por query`** ⚖️
  * *O que significa:* Cada pergunta tem exatamente 5 documentos com relevância média/aceitável (nota 1).
  * *Explicação matemática:* Definimos no mapa de afinidades que cada categoria possui exatamente **1 categoria vizinha correlacionada** (que também possui **5 frases** no banco). Logo, haverá sempre 5 documentos correlacionados disponíveis.
  
* **`Docs com grau 0 (média): 30.0 por query`** ❌
  * *O que significa:* Cada pergunta tem exatamente 30 documentos totalmente irrelevantes (nota 0).
  * *Explicação matemática:* De 40 documentos totais, removemos os 5 ideais (grau 2) e os 5 correlacionados (grau 1). Sobram exatamente **30 documentos fora do assunto** para cada busca realizada.

---

*💡 **Conclusão:** Esses resultados provam que a divisão de dados, o mapeamento de afinidades e o tamanho do dataset estão em perfeito equilíbrio matemático para começarmos a calcular as notas de exatidão de busca (como NDCG e MAP) de forma científica.*



In [ ]:
# Célula 10: Ground truth — queries de avaliação

# Mapa de afinidade temática (categoria → lista de categorias próximas)
afinidade_categorias = {
    'Crimes Patrimoniais':     ['Crimes Cibernéticos'],
    'Crimes Cibernéticos':     ['Crimes Patrimoniais'],
    'Crimes Contra a Vida':    ['Violência Doméstica'],
    'Violência Doméstica':     ['Crimes Contra a Vida'],
    'Direito Processual':      ['Investigação e Perícia'],
    'Investigação e Perícia':  ['Direito Processual'],
    'Crimes Administrativos':  ['Crimes Ambientais'],
    'Crimes Ambientais':       ['Crimes Administrativos'],
}

# 20 queries com vocabulário diferente das frases originais (evita overfitting lexical)
queries_avaliacao = [
    # Crimes Patrimoniais (3)
    {'query': 'documentos sobre apropriação indevida de bens alheios',          'categoria_alvo': 'Crimes Patrimoniais'},
    {'query': 'casos de assalto à mão armada em vias públicas',                  'categoria_alvo': 'Crimes Patrimoniais'},
    {'query': 'golpe financeiro com indução da vítima a erro',                  'categoria_alvo': 'Crimes Patrimoniais'},

    # Crimes Contra a Vida (2)
    {'query': 'tentativa de matar alguém com intenção',                          'categoria_alvo': 'Crimes Contra a Vida'},
    {'query': 'acidente de trânsito fatal por negligência do motorista',         'categoria_alvo': 'Crimes Contra a Vida'},

    # Direito Processual (3)
    {'query': 'pedido de soltura por prisão ilegal',                             'categoria_alvo': 'Direito Processual'},
    {'query': 'prazo de 24 horas após detenção do suspeito',                     'categoria_alvo': 'Direito Processual'},
    {'query': 'peça acusatória formulada pelo MP',                               'categoria_alvo': 'Direito Processual'},

    # Investigação e Perícia (3)
    {'query': 'exame técnico realizado pelos peritos no local',                  'categoria_alvo': 'Investigação e Perícia'},
    {'query': 'preservação dos vestígios coletados na cena do crime',            'categoria_alvo': 'Investigação e Perícia'},
    {'query': 'oitiva de testemunha no curso da investigação',                   'categoria_alvo': 'Investigação e Perícia'},

    # Crimes Administrativos (2)
    {'query': 'funcionário público recebendo propina',                           'categoria_alvo': 'Crimes Administrativos'},
    {'query': 'desvio de recursos públicos por agente do estado',                'categoria_alvo': 'Crimes Administrativos'},

    # Crimes Cibernéticos (3)
    {'query': 'invasão de sistema computacional para roubo de informações',      'categoria_alvo': 'Crimes Cibernéticos'},
    {'query': 'sequestro de arquivos digitais com pedido de resgate',            'categoria_alvo': 'Crimes Cibernéticos'},
    {'query': 'site falso criado para roubar credenciais bancárias',             'categoria_alvo': 'Crimes Cibernéticos'},

    # Crimes Ambientais (2)
    {'query': 'derrubada de árvores em área de preservação',                     'categoria_alvo': 'Crimes Ambientais'},
    {'query': 'poluição química de rios por indústria',                          'categoria_alvo': 'Crimes Ambientais'},

    # Violência Doméstica (2)
    {'query': 'descumprimento de ordem judicial de afastamento do agressor',     'categoria_alvo': 'Violência Doméstica'},
    {'query': 'tornozeleira eletrônica para autor de violência contra mulher',   'categoria_alvo': 'Violência Doméstica'},
]

# Enriquecer com categorias próximas
for q in queries_avaliacao:
    q['categorias_proximas'] = afinidade_categorias.get(q['categoria_alvo'], [])

df_queries = pd.DataFrame(queries_avaliacao)
print(df_queries)

print(f'📋 Ground truth construído: {len(df_queries)} queries')
print(f'   Categorias cobertas: {df_queries["categoria_alvo"].nunique()}')
print()
print('Distribuição por categoria-alvo:')
print(df_queries.groupby('categoria_alvo').size().to_string())

# Função utilitária: converter categoria de um doc em grau de relevância para uma query
def grau_relevancia(categoria_doc: str, query: dict) -> int:
    """Retorna 2 (mesma cat), 1 (cat próxima), 0 (não relacionada)."""
    if categoria_doc == query['categoria_alvo']:
        return 2
    if categoria_doc in query['categorias_proximas']:
        return 1
    return 0


# Pré-computar matriz de relevância (n_queries x n_docs) — usada em todas as métricas
relevancias = np.zeros((len(queries_avaliacao), len(frases)), dtype=int)
for i, q in enumerate(queries_avaliacao):
    for j, cat_doc in enumerate(categorias):
        relevancias[i, j] = grau_relevancia(cat_doc, q)

print(f'\n📊 Matriz de relevâncias: shape={relevancias.shape}')
print(f'   Docs com grau 2 (média): {(relevancias == 2).sum(axis=1).mean():.1f} por query')
print(f'   Docs com grau 1 (média): {(relevancias == 1).sum(axis=1).mean():.1f} por query')
print(f'   Docs com grau 0 (média): {(relevancias == 0).sum(axis=1).mean():.1f} por query')


## 📦 CÉLULA 11 — Hit Rate@K e Recall@K

**Hit Rate@K** (também chamado Recall binário): proporção de queries para as quais **pelo menos um** doc relevante aparece nas top-K posições.

$$ \text{HitRate@K} = \frac{1}{|Q|} \sum_{q \in Q} \mathbb{1}[\exists \text{ rel} \in \text{top-K}(q)] $$

**Recall@K**: fração dos documentos relevantes que foram efetivamente recuperados nas top-K posições.

$$ \text{Recall@K} = \frac{1}{|Q|} \sum_{q \in Q} \frac{|\text{rel} \cap \text{top-K}(q)|}{|\text{rel}(q)|} $$

**Quando usar cada K:**
- **K=1**: precisão de ranking — útil para chatbots que mostram só a melhor resposta
- **K=3-5**: cenário típico de RAG (LLM recebe 3-5 chunks de contexto)
- **K=10**: limite superior — útil quando há reranker em seguida



### 📊 Avaliação Prática de Retrieval: Entendendo as Métricas de RAG (Hit Rate@K e Recall@K)

Nesta etapa (Célula 11), saímos da teoria geométrica e entramos em um **cenário de teste de busca real**. Simulamos usuários reais fazendo perguntas jurídicas ao nosso sistema RAG e medimos numericamente a capacidade dos modelos de recuperar as respostas corretas dentro de diferentes limites de busca (definidos como **$K$**).

---

#### ⚖️ O que representa o parâmetro **$K$**?
O $K$ representa a "janela de exibição" de resultados. Por exemplo:
* **$K=1$**: Analisamos apenas o primeiro resultado trazido pela busca (o "Estou com sorte" do Google).
* **$K=5$**: Analisamos o Top-5 resultados que seriam injetados no contexto do prompt para a LLM.

---

#### 1️⃣ **Hit Rate@K (Taxa de Acerto)**
* **O que é:** Mede a proporção de buscas em que o modelo conseguiu colocar **pelo menos 1 documento relevante** dentro do Top-K.
* **Exemplo Prático:** Se o usuário busca por *"pedido de soltura por prisão ilegal"* e o seu sistema retorna 5 documentos, o Hit Rate@5 será de **100% (1.00)** se pelo menos um desses 5 documentos tratar de Direito Processual (o assunto correto). Se nenhum dos 5 for sobre esse tema, é um *Miss* (0.0). A média dessa taxa é calculada sobre as 20 queries de teste.
* **Foco Arquitetural:** Excelente métrica para sistemas onde o usuário final quer **uma resposta rápida e direta** (como chat de perguntas e respostas simples).

---

#### 2️⃣ **Recall@K (Fração de Recuperação)**
* **O que é:** Mede a proporção de documentos úteis recuperados em relação ao **total de documentos relevantes existentes no banco de dados**.
* **Exemplo Prático:** No nosso Ground Truth (Célula 10), sabemos que existem exatamente **10 documentos relevantes** no banco de dados para a busca realizada (5 da categoria exata e 5 da categoria vizinha correlacionada). Se no Top-5 resultados trazidos pela busca o modelo resgatou **4 desses documentos**, o **Recall@5** será de $\frac{4}{10} = \mathbf{40\% \ (0.40)}$.
* **Foco Arquitetural:** Crucial para RAGs jurídicos de **Pesquisa e Análise de Precedentes**, onde o advogado ou magistrado precisa ler *todas* ou a maioria das decisões parecidas para fundamentar uma peça, e não apenas uma única resposta isolada.

---

### 🧠 Como usar essas métricas para tomar decisões de Arquitetura RAG?

1. **A Escolha do Tamanho de Contexto ($K$ Ideal):**
   * Olhe para o gráfico de **Recall@K**. Se o Recall do modelo BGE-M3 atinge **`0.95` em $K=5$** e sobe muito pouco para **`0.98` em $K=10$**, significa que retornar 5 documentos já captura quase toda a informação relevante do banco. 
   * **Decisão de Engenharia:** Configure seu RAG para $K=5$. Isso reduzirá os custos de API da LLM pela metade e acelerará o tempo de resposta do sistema sem perder qualidade semântica!

2. **O Fenômeno do "Lost in the Middle":**
   * LLMs tendem a ignorar informações colocadas no meio de contextos muito longos. Um modelo com **Hit Rate@1** e **Hit Rate@3** muito altos permite que você use janelas de contexto menores ($K$ menor), posicionando a informação mais relevante no topo absoluto do prompt, onde a LLM presta mais atenção.

3. **Análise de Trade-off (Custo vs. Precisão):**
   * Se um modelo super leve via Ollama (como o `nomic-embed-text`) apresentar métricas de Hit Rate e Recall muito próximas às do BGE-M3, você pode decidir economizar em hardware pesado e optar pela infraestrutura local e veloz do Ollama.


In [ ]:
# Célula 11: Hit Rate@K e Recall@K (Adaptada para suportar Locais + Ollama dinamicamente)
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

def encode_queries(modelo, queries_list):
    """Encoda uma lista de strings em embeddings normalizados usando modelo local."""
    return modelo.encode(
        queries_list,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

def hit_rate_at_k(sims_matrix: np.ndarray, relev_matrix: np.ndarray, k: int) -> float:
    """Retorna proporção de queries com >=1 doc relevante (grau>=1) no top-K."""
    n_q = sims_matrix.shape[0]
    hits = 0
    for i in range(n_q):
        top_k = np.argsort(sims_matrix[i])[::-1][:k]
        if (relev_matrix[i, top_k] >= 1).any():
            hits += 1
    return hits / n_q

def recall_at_k(sims_matrix: np.ndarray, relev_matrix: np.ndarray, k: int) -> float:
    """Recall@K considerando docs com relevância >=1 como relevantes."""
    n_q = sims_matrix.shape[0]
    recalls = []
    for i in range(n_q):
        rels_total = (relev_matrix[i] >= 1).sum()
        if rels_total == 0:
            continue
        top_k = np.argsort(sims_matrix[i])[::-1][:k]
        rels_no_top = (relev_matrix[i, top_k] >= 1).sum()
        recalls.append(rels_no_top / rels_total)
    return float(np.mean(recalls))


# 1. UNIFICA OS RESULTADOS DE FORMA SEGURA
resultados_totais = {}
if 'resultados_locais' in globals() and resultados_locais:
    resultados_totais.update(resultados_locais)
if 'resultados_ollama' in globals() and resultados_ollama:
    resultados_totais.update(resultados_ollama)

# Lista de textos das queries do Ground Truth
text_queries = [q['query'] for q in queries_avaliacao]

sim_queries_docs = {}
query_embeddings = {}

# 2. GERA OS EMBEDDINGS DAS QUERIES DE FORMA INTELIGENTE (LOCAL OU OLLAMA)
for nome, res in resultados_totais.items():
    print(f'🔢 Gerando embeddings de queries para o modelo: {nome}...')
    
    if 'modelos_carregados' in globals() and nome in modelos_carregados:
        # Caso A: Modelo Local (Hugging Face)
        modelo = modelos_carregados[nome]
        q_emb = encode_queries(modelo, text_queries)
    else:
        # Caso B: Modelo do Ollama Server (Batch request para alto desempenho)
        try:
            response = requests.post(
                'http://localhost:11434/api/embed',
                json={'model': nome, 'input': text_queries},
                timeout=20
            )
            if response.status_code != 200:
                raise Exception(f"Erro na API do Ollama: {response.text}")
            
            q_emb = np.array(response.json().get('embeddings', []))
            
            # Normalização L2 para garantir a similaridade coseno
            norms = np.linalg.norm(q_emb, axis=1, keepdims=True)
            norms = np.where(norms == 0, 1e-9, norms)
            q_emb = q_emb / norms
            
        except Exception as e:
            print(f'   ❌ Erro com o modelo Ollama "{nome}": {e}')
            continue
            
    query_embeddings[nome] = q_emb
    # Cruza a similaridade entre as 20 queries e os 40 documentos salvos
    sim_queries_docs[nome] = cosine_similarity(q_emb, res['embeddings'])
    print(f'   ✅ Similaridades calculadas — shape {sim_queries_docs[nome].shape}')


# 3. CÁLCULO DAS MÉTRICAS PARA K = 1, 3, 5, 10
Ks = [1, 3, 5, 10]
metricas_hit_recall = {}

for nome in resultados_totais.keys():
    if nome not in sim_queries_docs:
        continue
    sims = sim_queries_docs[nome]
    metricas_hit_recall[nome] = {}
    for k in Ks:
        metricas_hit_recall[nome][f'Hit@{k}']    = hit_rate_at_k(sims, relevancias, k)
        metricas_hit_recall[nome][f'Recall@{k}'] = recall_at_k(sims, relevancias, k)

# Exibe tabela final no terminal
df_hit_recall = pd.DataFrame(metricas_hit_recall).T
print('\n📊 TABELA COMPARATIVA DE HIT RATE@K e RECALL@K')
print('=' * 80)
print(df_hit_recall.round(3).to_string())
print('=' * 80)


# 4. GRÁFICOS DE BARRAS AGRUPADOS COM DIMENSIONAMENTO DINÂMICO
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.set_style('whitegrid')

x = np.arange(len(Ks))
modelos_nomes = list(metricas_hit_recall.keys())
n_modelos = len(modelos_nomes)

# Define a largura ideal das barras baseado na quantidade de modelos ativos
width = 0.8 / n_modelos  
paleta = sns.color_palette("muted", n_modelos)

# Plot do Hit Rate
ax = axes[0]
for i, nome in enumerate(modelos_nomes):
    valores = [metricas_hit_recall[nome][f'Hit@{k}'] for k in Ks]
    # Posiciona a barra ajustando o deslocamento
    posicao = x + (i - (n_modelos - 1) / 2) * width
    ax.bar(posicao, valores, width, label=nome, color=paleta[i], edgecolor='black', linewidth=0.5)
    for j, v in enumerate(valores):
        ax.text(posicao[j], v + 0.015, f'{v:.2f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([f'K={k}' for k in Ks])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Hit Rate@K', fontsize=11, fontweight='bold')
ax.set_title('Hit Rate@K — Modelo deve recuperar ≥1 doc relevante', fontweight='bold', fontsize=12, pad=10)
ax.legend(title="Modelos Avaliados")
ax.grid(axis='y', alpha=0.3)

# Plot do Recall
ax = axes[1]
for i, nome in enumerate(modelos_nomes):
    valores = [metricas_hit_recall[nome][f'Recall@{k}'] for k in Ks]
    posicao = x + (i - (n_modelos - 1) / 2) * width
    ax.bar(posicao, valores, width, label=nome, color=paleta[i], edgecolor='black', linewidth=0.5)
    for j, v in enumerate(valores):
        ax.text(posicao[j], v + 0.015, f'{v:.2f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([f'K={k}' for k in Ks])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Recall@K', fontsize=11, fontweight='bold')
ax.set_title('Recall@K — Fração de docs relevantes recuperados', fontweight='bold', fontsize=12, pad=10)
ax.legend(title="Modelos Avaliados")
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('\n💡 INTERPRETAÇÃO ACADÊMICA DAS MÉTRICAS:')
print('   → Hit@1 alto = O modelo de embedding é extremamente confiável para trazer a resposta perfeita já na POSIÇÃO 1.')
print('   → Recall@5 ≈ 1.000 = Excelente sinal! Significa que o contexto completo do RAG cabe perfeitamente nos Top-5 resultados.')
print('   → Se o Recall@10 demorar para subir: O modelo sofre com dispersão semântica (mistura muitos documentos irrelevantes).')


## 📦 CÉLULA 12 — MRR (Mean Reciprocal Rank)

### 🎯 Mean Reciprocal Rank (MRR): Medindo a Precisão do Top-1 Resultado

Enquanto o *Hit Rate* mede se o resultado útil está dentro de uma janela de busca (Top-K) e o *Recall* mede a porcentagem de cobertura, o **MRR (Mean Reciprocal Rank)** responde a uma pergunta crucial de usabilidade de sistemas de busca ou assistentes de IA:

> **"O primeiro resultado trazido pelo modelo é realmente o melhor e o que eu precisava?"**

Em sistemas modernos de RAG, o primeiro documento trazido pela busca recebe o maior peso de atenção da LLM. O MRR nos ajuda a certificar matematicamente de que a resposta mais precisa está no topo absoluto do ranking de busca.

---

#### ⚖️ O Conceito Matemático: O Rank Recíproco (RR)
Para cada pergunta (query) testada, o código vasculha o ranking de resultados e calcula o score com base na posição do **primeiro documento relevante** encontrado:

* **1ª Posição (Top-1):** Score $1.00$ ($\frac{1}{1}$). *Cenário Perfeito.*
* **2ª Posição (Top-2):** Score $0.50$ ($\frac{1}{2}$).
* **3ª Posição (Top-3):** Score $0.33$ ($\frac{1}{3}$).
* **5ª Posição (Top-5):** Score $0.20$ ($\frac{1}{5}$).
* **Nenhuma posição encontrada:** Score $0.00$.

O **MRR (Mean Reciprocal Rank)** é a média aritmética simples dos RRs obtidos em todas as 20 perguntas do nosso Ground Truth.

* **MRR próximo a 1.00 (Ex: 0.92):** O modelo é espetacular. Em mais de 90% das buscas, o primeiro documento da lista já é o correto.
* **MRR próximo a 0.50:** O modelo é moderado. Em média, a primeira resposta correta costuma vir apenas na 2ª posição.

---

### 📊 Estrutura de Diagnóstico da Célula 12

O código gera duas visualizações complementares fundamentais para um engenheiro de RAG:

1. **O Boxplot (Gráfico de Caixa - Painel Esquerdo):**
   * Mostra a consistência dos modelos. Se a "caixa" de um modelo estiver espremida no topo (perto de 1.00), significa que o modelo é extremamente estável e consistente em suas buscas. Se a caixa for muito esticada ou tiver muitos pontos isolados na base (outliers), o modelo é instável e "chuta" com frequência.
   
2. **O Gráfico de Barras por Query (Painel Direito):**
   * Plota barras individuais de $q_1$ a $q_{20}$ comparando os modelos para cada pergunta de teste. Isso permite fazer um diagnóstico cirúrgico de qual tipo de sintaxe ou vocabulário jurídico faz cada modelo falhar.

---

### 🔍 Relatório de Diagnóstico: Identificação de Pontos Cegos
Na parte final da execução, o código imprime um **Relatório de Queries Problemáticas** (onde o $RR < 0.50$).
* **O que isso revela:** Mostra exatamente quais perguntas de teste fizeram os modelos colocarem a primeira resposta correta apenas na 3ª posição ou pior.
* **Ação Recomendada:** Perguntas com baixa nota indicam termos ambíguos ou conceitos jurídicos que exigem que você ajuste o dataset, melhore a redação das perguntas, aplique técnicas de expansão de queries (query expansion) ou opte por um modelo de embedding mais robusto (como o BGE-M3).


O MRR mede a **posição do primeiro doc relevante** em cada query:

$$ \text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q} $$

| Posição do 1º relevante | Contribuição |
|-------------------------|--------------|
| 1 | 1.000 |
| 2 | 0.500 |
| 3 | 0.333 |
| 5 | 0.200 |
| 10 | 0.100 |
| Não encontrado | 0.000 |

**Por que usar MRR:** Hit Rate diz *"achou ou não"*. MRR diz *"se achou, em que posição"*. Para RAG isso importa: rank 1 vs rank 5 muda a qualidade do contexto entregue ao LLM.


In [ ]:
# Célula 12: Mean Reciprocal Rank (Adaptado para Locais + Ollama dinamicamente)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def reciprocal_ranks(sims_matrix: np.ndarray, relev_matrix: np.ndarray) -> np.ndarray:
    """Retorna array (n_queries,) com 1/rank do primeiro doc relevante. 0 se nenhum."""
    n_q = sims_matrix.shape[0]
    rrs = np.zeros(n_q)
    for i in range(n_q):
        ranking = np.argsort(sims_matrix[i])[::-1]
        for rank, doc_idx in enumerate(ranking, start=1):
            if relev_matrix[i, doc_idx] >= 1:
                rrs[i] = 1.0 / rank
                break
    return rrs


# 1. UNIFICA OS RESULTADOS DE FORMA SEGURA
resultados_totais = {}
if 'resultados_locais' in globals() and resultados_locais:
    resultados_totais.update(resultados_locais)
if 'resultados_ollama' in globals() and resultados_ollama:
    resultados_totais.update(resultados_ollama)

mrr_por_modelo = {}
rrs_por_modelo = {}

print('📊 CÁLCULO DE MEAN RECIPROCAL RANK (MRR)')
print('=' * 75)

# 2. CALCULA O MRR PARA TODOS OS MODELOS ATIVOS
for nome in resultados_totais.keys():
    if nome not in sim_queries_docs:
        continue
    rrs = reciprocal_ranks(sim_queries_docs[nome], relevancias)
    rrs_por_modelo[nome] = rrs
    mrr_por_modelo[nome] = float(np.mean(rrs))
    print(f'✅ {nome}: MRR = {mrr_por_modelo[nome]:.4f}')

print('=' * 75)


# 3. CONFIGURAÇÃO DE PLOTAGEM DINÂMICA
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.set_style('whitegrid')

modelos_nomes = list(mrr_por_modelo.keys())
n_modelos = len(modelos_nomes)
paleta = sns.color_palette("muted", n_modelos)

# PAINEL 1: Boxplot (Distribuição de RRs)
ax = axes[0]
dados_box = [rrs_por_modelo[n] for n in modelos_nomes]
bp = ax.boxplot(dados_box, labels=modelos_nomes, patch_artist=True, widths=0.4)

# Colore os boxes usando a paleta dinâmica
for patch, cor in zip(bp['boxes'], paleta):
    patch.set_facecolor(cor)
    patch.set_alpha(0.7)
    
ax.set_ylabel('Reciprocal Rank (1/rank)', fontsize=11, fontweight='bold')
ax.set_title('Distribuição de Reciprocal Ranks por Query', fontweight='bold', fontsize=12, pad=10)
ax.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='rank 1 (ideal)')
ax.axhline(0.5, color='orange', linestyle='--', alpha=0.5, label='rank 2')
ax.legend()
ax.grid(axis='y', alpha=0.3)


# PAINEL 2: Comparação Query-a-Query Dinâmica
ax = axes[1]
x = np.arange(len(queries_avaliacao))
width = 0.8 / n_modelos  # Ajusta a largura das barras baseado no número de modelos

for i, nome in enumerate(modelos_nomes):
    posicao = x + (i - (n_modelos - 1) / 2) * width
    ax.bar(posicao, rrs_por_modelo[nome], width, label=nome, color=paleta[i], alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([f'q{i+1}' for i in range(len(queries_avaliacao))], rotation=0, fontsize=8)
ax.set_ylabel('Reciprocal Rank', fontsize=11, fontweight='bold')
ax.set_title('Reciprocal Rank por Query — Onde cada modelo falha', fontweight='bold', fontsize=12, pad=10)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


# 4. IDENTIFICAÇÃO DINÂMICA DE QUERIES PROBLEMÁTICAS (RR < 0.5 em algum modelo)
print('\n🔍 QUERIES PROBLEMÁTICAS (RR < 0.5 em algum modelo)')
print('=' * 90)
queries_problematicas_encontradas = False

for i, q in enumerate(queries_avaliacao):
    # Coleta o RR dessa query específica para todos os modelos analisados
    rrs_q = {nome: rrs_por_modelo[nome][i] for nome in modelos_nomes}
    
    if any(rr < 0.5 for rr in rrs_q.values()):
        queries_problematicas_encontradas = True
        rrs_str = ' | '.join(f'{n}: {r:.3f}' for n, r in rrs_q.items())
        print(f'  q{i+1} [{q["categoria_alvo"][:20]:20s}] "{q["query"][:50]}..."')
        print(f'        {rrs_str}')

if not queries_problematicas_encontradas:
    print("  🎉 Incrível! Nenhuma query problemática encontrada. Todos os modelos pontuaram RR >= 0.5 em todas as buscas!")

print('=' * 90)
print('\n💡 INTERPRETAÇÃO ACADÊMICA DO MRR:')
print('   → MRR = 1.0000 ⇒ Perfeito! O modelo sempre posiciona o documento correto no Top-1 absoluto.')
print('   → MRR = 0.5000 ⇒ Moderado. Em média, a primeira resposta útil aparece apenas no Top-2.')
print('   → Queries com RR < 0.5 indicam que a query jurídica possui ambiguidade ou vocabulário muito distante do banco.')


## 📦 CÉLULA 13 — NDCG@K (Normalized Discounted Cumulative Gain)

### 📈 NDCG@K - A Métrica de Ouro de Avaliação de Rankings

Enquanto as métricas anteriores (*Hit Rate* e *Recall*) avaliam a busca de forma binária (útil ou inútil), a indústria de busca utiliza o **NDCG (Ganho Cumulativo Descontado Normalizado)** como o padrão-ouro de avaliação. 

O NDCG entende que o conhecimento não é binário. No nosso Ground Truth (Célula 10), nós graduamos a relevância dos documentos em 3 níveis:
* **Grau 2 (Altamente Relevante):** Documentos da mesma categoria alvo.
* **Grau 1 (Parcialmente Relevante):** Documentos de categorias vizinhas correlacionadas.
* **Grau 0 (Irrelevante):** Documentos fora do assunto.

O NDCG avalia cientificamente se o modelo de embedding foi capaz de ordenar os documentos no ranking colocando **os excelentes no topo, os bons no meio e os ruins na base.**

---

#### 🧪 Como o NDCG é calculado por trás dos panos?

O NDCG combina três conceitos fundamentais da ciência da informação:

1. **Ganho Cumulativo (CG - Cumulative Gain):**
   * Mede a soma bruta de relevância dos documentos retornados. Trazer documentos de Grau 2 gera muito mais pontos para o modelo do que trazer documentos de Grau 1. Documentos de Grau 0 somam zero pontos.
   
2. **Desconto Logarítmico (DCG - Discounted Cumulative Gain):**
   * Os usuários raramente rolam a tela. Portanto, o modelo deve ser **penalizado** se colocar uma informação muito importante no final da lista.
   * O DCG aplica um decaimento logarítmico na pontuação baseado na posição do ranking. Um documento excelente (Grau 2) na **posição 1** dá pontuação máxima. Esse mesmo documento na **posição 10** tem sua pontuação de relevância cortada quase por completo.
   
3. **Normalização (NDCG):**
   * Para sabermos se a nota do modelo é boa, dividimos o DCG obtido pelo **DCG Ideal (IDCG)** — que é a pontuação máxima possível caso ordenássemos os mesmos elementos de forma perfeita (todos os Grau 2 primeiro, depois todos os Grau 1, e Grau 0 por último).
   * Isso calibra o resultado em uma escala estrita de **0.000 (pior busca possível) a 1.000 (busca perfeita)**.

---

### 💡 Como interpretar as notas do NDCG@K no seu RAG?

* **NDCG = 1.000:** Perfeito. O modelo classificou todos os documentos na ordem ideal de utilidade.
* **NDCG = 0.850:** Excelente. Significa que o modelo trouxe as informações certas no topo, cometendo pequenos deslizes de ordenação (como trocar a posição de um Grau 1 com um Grau 2).
* **NDCG < 0.500:** Insuficiente. O modelo está confundindo as nuances semânticas e jogando documentos irrelevantes no topo da lista.

#### ⚖️ Análise de Cauda (NDCG@3 vs NDCG@10):
* Se a nota do **NDCG@3** for menor que a do **NDCG@10**, significa que o modelo tem boa cobertura geral, mas costuma errar na ordem dos primeiros resultados (empurrando o documento principal mais para baixo).
* Em sistemas de RAG, o **NDCG@3** e **NDCG@5** são as métricas críticas, pois a maioria dos prompts injeta apenas de 3 a 5 fragmentos de texto no contexto das LLMs para evitar custos altos e alucinação.



$$ \text{DCG@K} = \sum_{i=1}^{K} \frac{2^{\text{rel}_i} - 1}{\log_2(i+1)} $$

$$ \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}} $$

**Por que NDCG é poderoso:**
1. Considera **graus** de relevância (não só "relevante/irrelevante")
2. Penaliza relevantes que aparecem em posições baixas (logaritmo)
3. Normalizado: 0 (pior ranking) a 1 (ranking ideal)

Aqui usamos escala **0-2** (definida na Célula 10):
- 2 = mesma categoria
- 1 = categoria tematicamente próxima
- 0 = não relacionada

> **NDCG é o gold standard em IR** — usado em benchmarks TREC, BEIR, MTEB.


In [ ]:
# Célula 13: NDCG@K usando sklearn (Adaptado para Locais + Ollama dinamicamente)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ndcg_score

# 1. UNIFICA OS RESULTADOS DE FORMA SEGURA
resultados_totais = {}
if 'resultados_locais' in globals() and resultados_locais:
    resultados_totais.update(resultados_locais)
if 'resultados_ollama' in globals() and resultados_ollama:
    resultados_totais.update(resultados_ollama)

ndcg_por_modelo = {}
Ks_ndcg = [3, 5, 10]

# 2. CALCULA O NDCG CIENTÍFICO PARA TODOS OS MODELOS ATIVOS
for nome in resultados_totais.keys():
    if nome not in sim_queries_docs:
        continue
    sims = sim_queries_docs[nome]
    ndcg_por_modelo[nome] = {}
    for k in Ks_ndcg:
        # A função ndcg_score espera y_true no formato (n_queries, n_docs)
        ndcg = ndcg_score(y_true=relevancias, y_score=sims, k=k)
        ndcg_por_modelo[nome][f'NDCG@{k}'] = ndcg

# Exibe a tabela comparativa no terminal
df_ndcg = pd.DataFrame(ndcg_por_modelo).T
print('📊 COMPARATIVO DE NDCG@K (Escala 0.0 a 1.0)')
print('=' * 65)
print(df_ndcg.round(4).to_string())
print('=' * 65)


# 3. GRÁFICO COMPARATIVO DE BARRAS DINÂMICO
fig, ax = plt.subplots(figsize=(12, 7))
sns.set_style('whitegrid')

x = np.arange(len(Ks_ndcg))
modelos_nomes = list(ndcg_por_modelo.keys())
n_modelos = len(modelos_nomes)

# Calcula a largura e paleta de cores dinamicamente
width = 0.8 / n_modelos
paleta = sns.color_palette("muted", n_modelos)

# Plota cada modelo lado a lado para cada K
for i, nome in enumerate(modelos_nomes):
    valores = [ndcg_por_modelo[nome][f'NDCG@{k}'] for k in Ks_ndcg]
    # Posiciona a barra com o deslocamento dinâmico calculado
    posicao = x + (i - (n_modelos - 1) / 2) * width
    bars = ax.bar(posicao, valores, width, label=nome,
                  color=paleta[i], edgecolor='black', linewidth=0.5)
    
    # Adiciona a nota numérica exata no topo de cada barra
    for j, v in enumerate(valores):
        ax.text(posicao[j], v + 0.015, f'{v:.3f}',
                ha='center', fontsize=9, fontweight='bold')

# Estética e formatação do gráfico
ax.set_xticks(x)
ax.set_xticklabels([f'K={k}' for k in Ks_ndcg], fontsize=11)
ax.set_ylabel('NDCG@K (Exatidão do Ranking)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_title('NDCG@K — Qualidade do Ranking com Relevância Graduada (0-2)',
             fontweight='bold', fontsize=14, pad=15)
ax.legend(title="Modelos Avaliados", fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('\n💡 DIRETRIZES DE INTERPRETAÇÃO DO NDCG:')
print('   → NDCG = 1.000 ⇒ Ranking Perfeito: Todas as respostas excelentes (Grau 2) vêm primeiro,')
print('                     seguidas pelas aceitáveis (Grau 1), e as irrelevantes na cauda.')
print('   → NDCG = 0.800 ⇒ Excelente qualidade de busca, ideal para colocar em produção.')
print('   → NDCG < 0.500 ⇒ Ruim. O modelo está trocando a ordem e empurrando lixo para o topo.')
print('\n📊 DIFERENÇA NDCG@3 vs NDCG@10:')
print('   - Se o NDCG@3 for alto, o seu RAG será extremamente preciso mesmo com poucos tokens de contexto.')
print('   - Se o NDCG@10 for muito superior ao NDCG@3, significa que o modelo localiza os documentos,')
print('     mas tem dificuldade de ordená-los com precisão no topo absoluto da lista.')




## 📦 CÉLULA 14 — UMAP com Queries Sobrepostas

**O que faz:** Aplica UMAP nos embeddings **combinados** (40 docs + 20 queries) e plota lado a lado para cada modelo.

**Como ler o gráfico:**
- 🔵 **Círculos** = documentos do dataset (cor = categoria)
- ⭐ **Estrelas** = queries (cor = categoria-alvo)
- **Linha verde** = query → top-1 doc recuperado (categoria correta)
- **Linha vermelha** = query → top-1 doc recuperado (categoria errada)

**O que esperar:**
- Modelo bom: cada estrela cai dentro do "cluster" da sua cor → linha verde curta
- Modelo ruim: estrela cai entre clusters ou no cluster errado → linha vermelha


In [ ]:
# Célula 14: UMAP com docs + queries sobrepostas (Adaptado para Locais + Ollama dinamicamente)
import numpy as np
import matplotlib.pyplot as plt
import umap

# 1. UNIFICA OS RESULTADOS DE FORMA SEGURA
resultados_totais = {}
if 'resultados_locais' in globals() and resultados_locais:
    resultados_totais.update(resultados_locais)
if 'resultados_ollama' in globals() and resultados_ollama:
    resultados_totais.update(resultados_ollama)

if not resultados_totais:
    print("❌ Nenhum resultado de embedding disponível para projetar na Célula 14.")
else:
    # 2. CONFIGURA OS SUBPLOTS COM BASE NA QUANTIDADE DE MODELOS
    fig, axes = plt.subplots(1, len(resultados_totais), figsize=(24, 11))
    if len(resultados_totais) == 1:
        axes = [axes]

    # Mapeamento de cores estáticas por categoria
    cor_por_cat = {cat: df[df['categoria'] == cat]['cor'].iloc[0]
                   for cat in df['categoria'].unique()}

    # 3. PROJETA E PLOTA CADA MODELO INDIVIDUALMENTE
    for idx, (nome, res) in enumerate(resultados_totais.items()):
        if nome not in query_embeddings or nome not in sim_queries_docs:
            print(f"⚠️ Embeddings de queries ou similaridades ausentes para o modelo '{nome}'. Pulando...")
            continue
            
        ax = axes[idx]

        # Concatena os embeddings dos documentos e das queries no mesmo espaço vetorial combinado
        doc_embs = res['embeddings']
        q_embs = query_embeddings[nome]
        combined = np.vstack([doc_embs, q_embs])

        print(f'\n🗺️  Calculando UMAP combinado para o modelo: {nome} (shape={combined.shape})...')

        # Redutor UMAP para 2 dimensões
        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=10,
            min_dist=0.15,
            metric='cosine',
            random_state=42
        )
        coords = reducer.fit_transform(combined)
        
        # Separa novamente as coordenadas 2D resultantes
        coords_docs = coords[:len(doc_embs)]
        coords_queries = coords[len(doc_embs):]

        # A) Plota os DOCUMENTOS por categoria como CÍRCULOS (●)
        for cat in df['categoria'].unique():
            mask = df['categoria'] == cat
            cor = df[mask]['cor'].iloc[0]
            pts = coords_docs[mask.values]
            ax.scatter(pts[:, 0], pts[:, 1], c=cor, s=160, alpha=0.6,
                       edgecolors='white', linewidth=1, label=f'doc · {cat}', zorder=3)

        # B) Plota as QUERIES de avaliação como ESTRELAS (★)
        sims_modelo = sim_queries_docs[nome]
        acertos = 0
        
        for i, q in enumerate(queries_avaliacao):
            cor_q = cor_por_cat[q['categoria_alvo']]
            
            # Identifica o Top-1 documento mais similar retornado pelo modelo
            top1_doc_idx = int(np.argmax(sims_modelo[i]))
            cat_top1 = categorias[top1_doc_idx]
            
            # Verde = Acertou a categoria | Vermelho = Errou a categoria
            cor_linha = '#27AE60' if cat_top1 == q['categoria_alvo'] else '#E74C3C'
            if cat_top1 == q['categoria_alvo']:
                acertos += 1

            # Desenha a linha de conexão da query (estrela) para o doc recuperado (círculo)
            ax.plot([coords_queries[i, 0], coords_docs[top1_doc_idx, 0]],
                    [coords_queries[i, 1], coords_docs[top1_doc_idx, 1]],
                    color=cor_linha, alpha=0.55, linewidth=1.5, zorder=2)

            # Plota a estrela representativa da pergunta
            ax.scatter(coords_queries[i, 0], coords_queries[i, 1],
                       c=cor_q, s=380, marker='*',
                       edgecolors='black', linewidth=1.2, zorder=5)
            
            # Identificador textual q1, q2... no gráfico
            ax.annotate(f'q{i+1}',
                        (coords_queries[i, 0], coords_queries[i, 1]),
                        textcoords='offset points', xytext=(6, 6),
                        fontsize=9, fontweight='bold', zorder=6)

        # Calcula a taxa de acerto Top-1 do modelo
        taxa_acerto = acertos / len(queries_avaliacao)
        ax.set_title(f'{nome}\nUMAP — Docs (●) + Queries (★) · Acerto top-1: {taxa_acerto:.0%}',
                     fontsize=13, fontweight='bold', pad=10)

        # Configura legenda sem duplicar os marcadores
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip([l.replace('doc · ', '') for l in labels], handles))
        ax.legend(unique.values(), unique.keys(),
                  loc='upper right', bbox_to_anchor=(1.22, 1), fontsize=9)
        
        ax.set_xlabel('Dimensão UMAP 1')
        ax.set_ylabel('Dimensão UMAP 2')

    plt.tight_layout()
    # Salva o mapa semântico na pasta local
    plt.savefig('mapa_semantico_umap_docs_queries.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n💡 DIRETRIZES DE INTERPRETAÇÃO DO GRÁFICO UMAP DE BUSCA:')
    print('   → Estrela (★) dentro do cluster de mesma cor + Linha VERDE = Busca Perfeita!')
    print('   → Linha VERMELHA = Falha Crítica. O modelo buscou uma pergunta e o primeiro resultado')
    print('     retornado pertencia a um tema completamente errado.')
    print('   → Estrela na fronteira semântica = Indica uma pergunta muito abrangente ou ambígua.')



## 📦 CÉLULA 15 — Histograma de Similaridade: Positivos vs Negativos

**O que faz:** Para cada query, separa as similaridades cosseno em duas distribuições:
- **Positivos:** similaridade com docs da MESMA categoria (relevância ≥ 1)
- **Negativos:** similaridade com docs de outras categorias (relevância = 0)

**Por que isso é poderoso:**
- Um bom modelo de embedding deve ter **clara separação** entre as duas distribuições
- Quanto **menor a sobreposição**, melhor o modelo discrimina relevantes/irrelevantes
- Métricas derivadas:
  - **Δμ = μ_pos − μ_neg** (quanto maior, melhor)
  - **AUC-ROC**: probabilidade de um positivo aleatório ter similaridade > um negativo aleatório


In [ ]:
# Célula 15: Histograma Positivos vs Negativos + AUC-ROC

from sklearn.metrics import roc_auc_score, roc_curve

fig, axes = plt.subplots(2, len(resultados_embedding), figsize=(18, 11))
if len(resultados_embedding) == 1:
    axes = axes.reshape(2, 1)

auc_por_modelo = {}
delta_mu_por_modelo = {}

for idx, nome in enumerate(modelos_nomes):
    sims = sim_queries_docs[nome]

    # Coletar pares pos/neg (achatado)
    pares_pos = []  # (query_idx, doc_idx, sim)
    pares_neg = []
    for i in range(len(queries_avaliacao)):
        for j in range(len(frases)):
            if relevancias[i, j] >= 1:
                pares_pos.append(sims[i, j])
            else:
                pares_neg.append(sims[i, j])

    pares_pos = np.array(pares_pos)
    pares_neg = np.array(pares_neg)

    # AUC-ROC: positivos têm sim maior?
    y_true = np.concatenate([np.ones_like(pares_pos), np.zeros_like(pares_neg)])
    y_score = np.concatenate([pares_pos, pares_neg])
    auc = roc_auc_score(y_true, y_score)
    auc_por_modelo[nome] = auc

    delta_mu = float(pares_pos.mean() - pares_neg.mean())
    delta_mu_por_modelo[nome] = delta_mu

    # --- Painel superior: histograma sobreposto ---
    ax = axes[0, idx]
    ax.hist(pares_neg, bins=30, alpha=0.6, color='#95A5A6',
            label=f'Negativos (n={len(pares_neg)})', edgecolor='white')
    ax.hist(pares_pos, bins=30, alpha=0.7, color='#27AE60',
            label=f'Positivos (n={len(pares_pos)})', edgecolor='white')
    ax.axvline(pares_neg.mean(), color='#7F8C8D', linestyle='--', linewidth=2,
               label=f'μ_neg = {pares_neg.mean():.3f}')
    ax.axvline(pares_pos.mean(), color='#229954', linestyle='--', linewidth=2,
               label=f'μ_pos = {pares_pos.mean():.3f}')
    ax.set_title(f'{nome}\nΔμ = {delta_mu:.3f} | AUC = {auc:.3f}',
                 fontweight='bold', fontsize=13)
    ax.set_xlabel('Similaridade cosseno')
    ax.set_ylabel('Frequência')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)

    # --- Painel inferior: curva ROC ---
    ax = axes[1, idx]
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ax.plot(fpr, tpr, color=cores_modelos[idx], linewidth=2.5,
            label=f'ROC (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.6, label='Aleatório (AUC=0.5)')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Curva ROC — {nome}', fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📊 SEPARABILIDADE POSITIVOS vs NEGATIVOS')
print('=' * 60)
for nome in modelos_nomes:
    print(f'  {nome}:')
    print(f'     Δμ (média pos − média neg): {delta_mu_por_modelo[nome]:.4f}')
    print(f'     AUC-ROC:                     {auc_por_modelo[nome]:.4f}')

print('\n💡 INTERPRETAÇÃO:')
print('   → AUC=1.0  ⇒ separação perfeita (modelo nunca confunde relevante/irrelevante)')
print('   → AUC=0.5  ⇒ aleatório (modelo inútil)')
print('   → AUC>0.85 ⇒ excelente para produção')
print('   → Δμ alto + variância baixa = embeddings discriminativos')


## 📦 CÉLULA 16 — Dashboard Consolidado de Métricas

Reunindo todas as métricas calculadas em uma única visão:
- Tabela mestre numérica
- **Radar chart** (spider plot) — visão geométrica das forças/fraquezas de cada modelo
- Recomendação automática com base no número de vitórias por métrica


In [ ]:
# Célula 15: Histograma Positivos vs Negativos + AUC-ROC (Adaptado para Locais + Ollama dinamicamente)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve

# 1. UNIFICA OS RESULTADOS DE FORMA SEGURA
resultados_totais = {}
if 'resultados_locais' in globals() and resultados_locais:
    resultados_totais.update(resultados_locais)
if 'resultados_ollama' in globals() and resultados_ollama:
    resultados_totais.update(resultados_ollama)

modelos_nomes = list(resultados_totais.keys())
n_modelos = len(modelos_nomes)

if not resultados_totais:
    print("❌ Nenhum resultado de embedding disponível para analisar na Célula 15.")
else:
    # 2. CONFIGURA O GRID DE SUBPLOTS DINAMICAMENTE
    fig, axes = plt.subplots(2, n_modelos, figsize=(20, 12))
    sns.set_style('whitegrid')
    if n_modelos == 1:
        axes = axes.reshape(2, 1)

    auc_por_modelo = {}
    delta_mu_por_modelo = {}
    
    # Paleta de cores dinâmica para as curvas ROC
    paleta = sns.color_palette("muted", n_modelos)

    # 3. COMPUTA E DESENHA O HISTOGRAMA E A CURVA ROC PARA CADA MODELO
    for idx, nome in enumerate(modelos_nomes):
        if nome not in sim_queries_docs:
            print(f"⚠️ Similaridades ausentes para '{nome}'. Pulando...")
            continue
            
        sims = sim_queries_docs[nome]

        # Coleta todas as similaridades de pares positivos (grau >= 1) e negativos (grau == 0)
        pares_pos = []  
        pares_neg = []
        for i in range(len(queries_avaliacao)):
            for j in range(len(frases)):
                if relevancias[i, j] >= 1:
                    pares_pos.append(sims[i, j])
                else:
                    pares_neg.append(sims[i, j])

        pares_pos = np.array(pares_pos)
        pares_neg = np.array(pares_neg)

        # Cálculo do AUC-ROC
        y_true_roc = np.concatenate([np.ones_like(pares_pos), np.zeros_like(pares_neg)])
        y_score_roc = np.concatenate([pares_pos, pares_neg])
        auc = roc_auc_score(y_true_roc, y_score_roc)
        auc_por_modelo[nome] = auc

        # Delta Mu (Distância entre as médias)
        delta_mu = float(pares_pos.mean() - pares_neg.mean())
        delta_mu_por_modelo[nome] = delta_mu

        # ─── Painel Superior: Histograma de Frequência Cosseno Sobreposto ───
        ax = axes[0, idx]
        ax.hist(pares_neg, bins=30, alpha=0.6, color='#95A5A6',
                label=f'Negativos (n={len(pares_neg)})', edgecolor='white')
        ax.hist(pares_pos, bins=30, alpha=0.7, color='#27AE60',
                label=f'Positivos (n={len(pares_pos)})', edgecolor='white')
        ax.axvline(pares_neg.mean(), color='#7F8C8D', linestyle='--', linewidth=2,
                   label=f'μ_neg = {pares_neg.mean():.3f}')
        ax.axvline(pares_pos.mean(), color='#229954', linestyle='--', linewidth=2,
                   label=f'μ_pos = {pares_pos.mean():.3f}')
        ax.set_title(f'{nome}\nΔμ = {delta_mu:.3f} | AUC = {auc:.3f}',
                     fontweight='bold', fontsize=13, pad=10)
        ax.set_xlabel('Similaridade cosseno')
        ax.set_ylabel('Frequência')
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(alpha=0.3)

        # ─── Painel Inferior: Curva ROC do Modelo ───
        ax = axes[1, idx]
        fpr, tpr, _ = roc_curve(y_true_roc, y_score_roc)
        ax.plot(fpr, tpr, color=paleta[idx], linewidth=2.5,
                label=f'ROC (AUC = {auc:.3f})')
        ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.6, label='Aleatório (AUC=0.5)')
        ax.set_xlabel('False Positive Rate (Taxa Falsos Positivos)')
        ax.set_ylabel('True Positive Rate (Taxa Verdadeiros Positivos)')
        ax.set_title(f'Curva ROC — {nome}', fontweight='bold', fontsize=12, pad=10)
        ax.legend(loc='lower right')
        ax.grid(alpha=0.3)

    plt.tight_layout()
    # Salva o gráfico estatístico localmente
    plt.savefig('separabilidade_auc_roc_modelos.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 4. EXIBIÇÃO DA TABELA CONSOLIDADA NO TERMINAL
    print('\n📊 TABELA DE SEPARABILIDADE POSITIVOS vs NEGATIVOS')
    print('=' * 65)
    for nome in modelos_nomes:
        if nome not in delta_mu_por_modelo:
            continue
        print(f'🤖 Modelo: {nome}')
        print(f'   👉 Δμ (Média Positivos − Média Negativos): {delta_mu_por_modelo[nome]:.4f}')
        print(f'   👉 AUC-ROC (Poder de Classificação):      {auc_por_modelo[nome]:.4f}')
        print('-' * 65)

    print('\n💡 INTERPRETAÇÃO ACADÊMICA:')
    print('   → AUC = 1.0000 ⇒ Perfeito! O modelo separa de forma impecável o que é relevante do irrelevante.')
    print('   → AUC = 0.5000 ⇒ Inútil. A busca do modelo equivale a chover palpites aleatórios (cara ou coroa).')
    print('   → AUC > 0.8500 ⇒ Excelente! O modelo é robusto e está pronto para ir para produção em RAG.')
    print('   → Δμ (Delta Mu) Alto: Indica que as duas curvas do histograma estão distantes uma da outra.')
    print('     Quanto menos sobreposição (área misturada) entre o cinza e o verde, menor é o ruído de busca.')


## 📦 CÉLULA 17 — 🚀 Outros Métodos de Avaliação Sugeridos

Além das 3 métricas exploradas (Hit Rate, MRR, NDCG) e das 2 visualizações (UMAP, Histograma), existem **muitas outras** técnicas relevantes para avaliação de embeddings em RAG. Algumas merecem destaque:

### 🎯 Métricas adicionais de retrieval

| Método | Quando aplicar | Complementa |
|--------|----------------|-------------|
| **MAP** (Mean Average Precision) | Quando há múltiplos relevantes por query e a ordem entre eles importa | NDCG |
| **Precision@K** | Cenário em que cada falso positivo no top-K tem custo (UI mostra todos) | Recall@K |
| **F1@K** | Quando precisa balancear precisão e recall em um único número | Hit@K |
| **R-Precision** | Precision na posição = número de docs relevantes | Recall@K |

### 📊 Métricas sem queries (clustering interno do dataset)

| Método | O que mede | Biblioteca |
|--------|------------|------------|
| **Silhouette Score** | Quão bem cada doc está agrupado em sua categoria vs outras | `sklearn.metrics.silhouette_score` |
| **Davies-Bouldin Index** | Razão de dispersão intra/inter cluster (menor = melhor) | `sklearn.metrics.davies_bouldin_score` |
| **Calinski-Harabasz** | Razão de variância inter/intra cluster (maior = melhor) | `sklearn.metrics.calinski_harabasz_score` |
| **Triplet Accuracy** | % de triplets (âncora, +, −) em que d(â,+) < d(â,−) | manual |

### 🏆 Benchmarks padronizados

| Benchmark | Cobertura | URL |
|-----------|-----------|-----|
| **MTEB** | 56 datasets, 8 tarefas (retrieval, classification, clustering, ...) | huggingface.co/spaces/mteb/leaderboard |
| **BEIR** | 18 datasets de retrieval zero-shot | github.com/beir-cellar/beir |
| **MIRACL** | 18 idiomas (inclui português) | github.com/project-miracl/miracl |
| **Mr.TyDi** | Retrieval cross-lingual | github.com/castorini/mr.tydi |

### ⚡ Métricas operacionais (produção)

- **Latência p50 / p95 / p99** (ms para encodar 1 query)
- **Throughput** (queries/segundo, batch size ótimo)
- **VRAM / RAM consumida** durante encoding
- **Tamanho do modelo em disco** (MB) e tempo de cold start
- **Custo por 1 milhão de queries** (se hospedado na nuvem)

### 🔬 Análises avançadas

1. **Análise por categoria/subdomínio** — quebrar TODAS as métricas por categoria jurídica para identificar pontos cegos (ex.: modelo forte em Direito Processual mas fraco em Cibernéticos)
2. **Robustez a typos** — injetar erros de digitação nas queries e medir queda no Recall@5
3. **Robustez a paráfrases** — gerar paráfrases com LLM e medir consistência do top-K
4. **Recall@K com Reranker (BGE-Reranker-v2-M3)** — preview da Aula 3 (#T03), mede ganho do cross-encoder sobre o bi-encoder
5. **Recall vs tamanho de chunk** — testar diferentes chunk sizes (256, 512, 1024) e medir trade-off
6. **Análise de hard negatives** — identificar os pares (query, doc_negativo) com maior similaridade falsa (candidatos a fine-tuning)
7. **Comparação com SPLADE / BM25** — preparação para Hybrid Search (#T04, Aula 4)
8. **Stability** — encodar a mesma query várias vezes e medir variância (idealmente zero)

### 📋 RAGAS (a partir da Aula 5)

Quando integrarmos LLMs ao pipeline, RAGAS adicionará 4 métricas obrigatórias:
- **Faithfulness** ≥ 0.80
- **Answer Relevancy** ≥ 0.75
- **Context Recall** ≥ 0.70
- **Context Precision** ≥ 0.70


##  Próximo Passo

Com os fundamentos dominados, avance para o **EXEMPLO_Pipeline_RAG_Minimo.ipynb**
para ver todos os componentes integrados: corpus → embedding → Ollama.

